# Automation and Affordability in U.S. Counties

Is the task composition of local work associated with what residents can afford?

Chris Bell  
Julian Pacheco

In [1]:
import os, sys
os.environ["R_HOME"]=os.path.join(sys.prefix, "lib", "R")
%load_ext rpy2.ipython
import rpy2.robjects as ro
from great_tables import style, loc
ro.r('''
suppressMessages({
  library(ggplot2)
  library(patchwork)
  library(scales)
})

BLUE <- "#2A78D6"
ORANGE <- "#EB6834"
GREEN <- "#1BAF7A"
PINK <- "#C2255C"
GREY <- "#8F8D87"

# Reserve colors, available if a new semantic group needs its own. Raw hex,
# unvalidated. RED is in active use for the fig-diagnostics loess trend line.
RED <- "#F21A00"
PURPLE <- "#35274A"
FOREST <- "#0B775E"
BROWN <- "#79402E"
MAGENTA <- "#E6A0C4"

# Task groups and purchasing power. Scoped to these five categorical series
# only; lightness varies within each hue family so the same-family pairs stay
# distinguishable under red-green color vision deficiency.
PP_TEAL <- "#1E8E99"
RC_TEAL <- "#51C3CC"
RM_TEAL <- "#006666"
NRC_ORANGE <- "#FFAD65"
NRM_ORANGE <- "#993F00"
TASK_COLORS <- c(
  "Purchasing Power" = PP_TEAL,
  "Routine Cognitive" = RC_TEAL,
  "Routine Manual" = RM_TEAL,
  "Non-Routine Cognitive" = NRC_ORANGE,
  "Non-Routine Manual" = NRM_ORANGE
  )

# Dollar axis labels, used by every figure reporting purchasing power.
dollar_axis <- function(v) ifelse(is.na(v), "", ifelse(v == 0, "$0",
  sprintf("%s$%s", ifelse(v < 0, "-", ""), formatC(abs(v), format="d", big.mark=","))))

theme_set(
  theme_minimal(base_size=9.5) +
    theme(
      panel.background=element_rect(fill=NA, color=NA),
      plot.background=element_rect(fill=NA, color=NA),
      panel.grid.major=element_line(color="#8F8D87", linewidth=0.3),
      panel.grid.minor=element_blank(),
      plot.title=element_text(face="bold", size=13, color="#0B0B0B", hjust=0.5, margin=margin(b=4)),
      plot.subtitle=element_text(size=8.5, color="#52514E", margin=margin(b=8)),
      strip.background=element_blank(),
      strip.text=element_text(face="bold", size=9, color="#0B0B0B"),
      axis.line=element_line(color="#0B0B0B", linewidth=0.4),
      axis.ticks=element_blank(),
      axis.text=element_text(size=9, color="#0B0B0B", face="bold"),
      axis.title=element_text(size=9.5, color="#0B0B0B"),
      legend.position="bottom",
      legend.title=element_blank(),
      legend.text=element_text(size=8.5, color="#0B0B0B"),
      legend.key=element_blank(),
      plot.margin=margin(10, 14, 8, 10)
    )
)
''')

def style_table(gt):
    """Shared table typography, applied to every GT table in the report so table
    text reads clearly smaller than body prose (great_tables defaults to 16px,
    matching body text)."""
    return gt.tab_options(
        table_font_size="12.5px",
        table_background_color="transparent",
        heading_title_font_size="14px",
        heading_subtitle_font_size="12px",
        column_labels_font_size="12.5px",
        column_labels_font_weight="bold",
        source_notes_font_size="10.5px",
    ).tab_style(
        style=style.text(style="italic"),
        locations=loc.source_notes(),
    )

R callback write-console: In addition:   
R callback write-console: Warning message:
  
R callback write-console: package ‘ggplot2’ was built under R version 4.5.3 
  

# Introduction

According to Gallup, 22 percent of American workers worried technology would make their job obsolete in 2023, up seven percentage points from 2021 ([Saad 2023](#ref-Gallup2023)). Concerns about automation were evident years earlier, when in 2017 the Pew Research Center reported that Americans expected automation to displace workers, although only 30 percent considered it likely that their own job would be replaced within their lifetime ([Smith and Anderson 2017](#ref-Pew2017)). These concerns reflect subjective expectations about future disruption rather than empirical evidence of current labor market conditions.The primary concern was the impact of job disruption on the economic security of households and the communities they resided in.

Couch and Placzek ([**CouchPlaczek2010?**](#ref-CouchPlaczek2010)) found that displaced workers experienced earnings losses of more than 30 percent initially, with losses remaining as high as 15 percent six years later. The extent of these losses varied depending on local prices, making purchasing power a valuable metric for assessing these concerns. While earnings captured the losses experienced by displaced workers, our study used median household income to describe household resources at the county level. Due to varying local prices, the same income could support different living standards in different places ([Moretti 2013](#ref-Moretti2013)). To capture this difference, purchasing power was adjusted by county median household income using Bureau of Economic Analysis Regional Price Parities. The study then explored the relationship between purchasing power and county task groups.

The framework we build upon is based on three studies. The first study, conducted by Autor, Levy, and Murnane ([2003](#ref-Autor2003)), demonstrated that computers replace routine tasks and enhance non-routine ones, enabling us to identify occupations susceptible to automation. Applying this concept to local labor markets, Autor and Dorn ([2013](#ref-AutorDorn2013)) observed polarization in routine-intensive sectors. In these areas, employment growth was observed at both the high and low ends of the wage distribution, while employment in the middle wage range declined. Subsequently, Acemoglu and Autor ([2011](#ref-AcemogluAutor2011)) generalized the argument into a model where tasks serve as the production unit, and skill groups compete to supply them.

Nevertheless, none of this research directly addressed purchasing power at the county level. These studies reported outcomes in unadjusted wages and employment counts rather than in what those wages could buy locally. They also measured these outcomes at a broader labor market geography than the county. Earlier studies primarily focused on employment and wages, leaving the question unanswered whether purchasing power varied with task groups in fixed dollar amounts or in proportion.

In this study, we investigate the relationship between county task groups and purchasing power, after controlling for poverty and unemployment. We utilize a county-level panel dataset spanning from 2008 to 2023, excluding the year 2020. This dataset comprises 840 counties and 11,983 observations. All input data originates from federal sources that are published annually at the county level, ensuring the reproducibility of our analysis using publicly available data. Notably, these counties collectively represent approximately 84 percent of the United States population, as reported by the Census Bureau in 2023 ([U.S. Census Bureau 2023b](#ref-census_popest2023)). In 2023, the counties exhibited an average of 39 percent routine task content, with the middle range of task content varying between 35 and 43 percent. County employment is categorized into four task groups, as illustrated in <a href="#fig-taskframework" class="quarto-xref">Figure 1</a> and further described in <a href="#sec-data" class="quarto-xref">Section 3</a>, along with the overall construction of the panel.

In [2]:
%%R -w 8.5 -h 6 -u in -r 150 -b transparent
quad_df <- data.frame(
  group = c("Non-Routine Cognitive", "Non-Routine Manual", "Routine Cognitive", "Routine Manual"),
  col = c(2, 2, 1, 1),
  row = c(2, 1, 2, 1),
  examples = c("Management, engineering,\nteaching", "Food service, care work,\ncleaning",
               "Clerical, sales,\nclaims processing", "Production, assembly,\nmachine operation"),
  is_reference = c(TRUE, FALSE, FALSE, FALSE)
)
quad_df$group_label <- gsub(" ", "\n", quad_df$group)
quad_df$linetype <- ifelse(quad_df$is_reference, "dashed", "solid")
quad_df$linewidth <- ifelse(quad_df$is_reference, 1.3, 0.9)

box_w <- 0.42; box_h <- 0.42

col_headers <- data.frame(
  col = c(1, 2),
  label = c("Routine", "Non-Routine")
)
row_headers <- data.frame(
  row = c(2, 1),
  label = c("Cognitive", "Manual")
)

ggplot(quad_df) +
  geom_rect(aes(xmin=col-box_w, xmax=col+box_w, ymin=row-box_h, ymax=row+box_h,
                color=group, linetype=linetype, linewidth=linewidth),
            fill="#FCFCFB") +
  geom_text(aes(x=col, y=row+0.14, label=group_label, color=group),
            fontface="bold", size=4.6, lineheight=0.95) +
  geom_text(aes(x=col, y=row-0.20, label=examples), color="#52514E", size=3.1, lineheight=1.0) +
  geom_text(data=col_headers, aes(x=col, y=2.65, label=label),
            fontface="bold", size=4.3, color="#0B0B0B", inherit.aes=FALSE) +
  geom_text(data=row_headers, aes(x=0.52, y=row, label=label),
            hjust=1, size=3.6, color="#0B0B0B", lineheight=0.95, inherit.aes=FALSE) +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  scale_linetype_identity() +
  scale_linewidth_identity() +
  coord_cartesian(xlim=c(0.05, 2.55), ylim=c(0.5, 2.85), clip="off") +
  theme_void() +
  theme(plot.margin=margin(10, 10, 10, 10),
        plot.background=element_rect(fill=NA, color=NA),
        plot.title=element_text(face="bold", size=13, hjust=0.5)) +
  labs(title="The Four Task Groups")

We estimated a panel regression model with year indicators and standard errors clustered by county to account for dependence among repeated observations of the same county. Diagnostic checks led to a log respecification, while random forest and neural network models tested whether additional flexibility captured structure missed by the linear model. Our primary finding was that a one percentage point shift from non-routine cognitive work to non-routine manual work was associated with approximately \$846 less purchasing power (95% CI: ±\$38 at the panel mean). Task groups therefore provided information about local economic conditions beyond poverty and unemployment rates. This additional information could help policymakers and economic development agencies identify differences across counties that conventional measures alone did not capture. <a href="#sec-analysis" class="quarto-xref">Section 4</a> describes the modeling strategy and diagnostics.

# Background

The consequences of automation differ across places because every place does not depend on the same kinds of work. Autor, Levy, and Murnane ([2003](#ref-Autor2003)) provided the framework for explaining why. Rather than treating occupations as either automated or not automated, they argued that technology replaces specific tasks within occupations. Routine tasks are easier to translate into explicit rules, while work that depends on judgment, adaptation, or direct interaction is harder to automate. This distinction produces the four task groups used in this study: routine cognitive, routine manual, non-routine cognitive, and non-routine manual.

Once work is understood through tasks, differences between local economies become measurable. Autor and Dorn ([2013](#ref-AutorDorn2013)) showed that these differences persist across local labor markets and are associated with long term employment polarization. Areas with more routine work saw middle wage employment decline while employment grew at the lower and upper ends of the wage distribution. Their occupation taxonomy provides the basis for assigning occupations to task groups here. Acemoglu and Autor ([2011](#ref-AcemogluAutor2011)) explained why those differences matter economically, modeling tasks as the unit of production over which workers with different skills compete. Technological change shifts which workers hold an advantage, which makes the consequences of technology depend partly on the kinds of work a local economy contains.

Task groups are only one part of a county’s economic conditions. Counties are also described by poverty, unemployment, and median household income. Poverty identifies households with limited resources, unemployment reflects access to work, and median household income describes what households receive. None of the three alone describes the conditions residents face.

Median household income is used because it better represents the income of a typical household than the mean, which is more sensitive to high incomes in a positively skewed distribution ([Chiripanhura 2011](#ref-Chiripanhura2011)). It describes what households receive rather than what they can buy. Because local prices differ, the same income supports different living standards in different places. Purchasing power combines the two by adjusting income with Regional Price Parities, which can change how counties compare rather than shifting them all by the same amount.

<a href="#fig-county-context" class="quarto-xref">Figure 2</a> maps all four measures across counties. Poverty (Panel A) and unemployment (Panel B) mark different concentrations of distress. Median household income (Panel C) shows where resources are highest and lowest before prices are taken into account. Regional Price Parity (Panel D) shows where those prices are high or low. The patterns do not align. A county that looks strong under one measure can look weaker under another, which is why no single conventional measure summarizes local economic conditions.

In [3]:
import os
import pandas as pd
import geopandas as gpd
from sqlalchemy import create_engine

engine = create_engine(
    os.environ["AUTORACK_URL"],
    pool_pre_ping=True,
    pool_recycle=300
)

In [4]:
context_year = 2023
context_df = pd.read_sql("""
    select cb.county_fips, cb.poverty_rate, cb.unemployment_rate,
           cb.median_household_income, ca.affordability_salary as purchasing_power
    from county_baseline cb
    join county_affordability ca
      on ca.county_fips = cb.county_fips and ca.year = cb.year
    where cb.year = %(yr)s
""", engine, params={"yr": context_year})
context_df["fips"] = context_df["county_fips"].astype(str).str.zfill(5)
# RPP recovered from the already computed purchasing power measure (@eq-afford, inverted):
# purchasing power = income / (RPP / 100), so RPP = income / purchasing power * 100.
context_df["rpp"] = context_df["median_household_income"] / context_df["purchasing_power"] * 100

gdf_context = gpd.read_file(
    "https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip"
)
gdf_context = gdf_context.rename(columns={"GEOID": "fips"})
merged_context = gdf_context.merge(
    context_df[["fips", "poverty_rate", "unemployment_rate",
                "median_household_income", "rpp"]],
    on="fips", how="left"
)
merged_context = merged_context[
    ~merged_context["STATEFP"].isin(["02", "15", "60", "66", "69", "72", "78"])
].copy()

pov_ctx_vmin = float(context_df["poverty_rate"].quantile(0.02))
pov_ctx_vmax = float(context_df["poverty_rate"].quantile(0.98))
unemp_ctx_vmin = float(context_df["unemployment_rate"].quantile(0.02))
unemp_ctx_vmax = float(context_df["unemployment_rate"].quantile(0.98))
inc_ctx_vmin = float(context_df["median_household_income"].quantile(0.02))
inc_ctx_vmax = float(context_df["median_household_income"].quantile(0.98))
rpp_ctx_vmin = float(context_df["rpp"].quantile(0.02))
rpp_ctx_vmax = float(context_df["rpp"].quantile(0.98))

os.makedirs("output", exist_ok=True)
merged_context[["fips", "poverty_rate", "unemployment_rate",
                "median_household_income", "rpp", "geometry"]].to_file(
    "output/county_context.geojson", driver="GeoJSON"
)

In [5]:
%%R -i pov_ctx_vmin -i pov_ctx_vmax -i unemp_ctx_vmin -i unemp_ctx_vmax -i inc_ctx_vmin -i inc_ctx_vmax -i rpp_ctx_vmin -i rpp_ctx_vmax -w 10 -h 9 -u in -r 150 -b transparent
suppressMessages(library(sf))

context_sf <- st_read("output/county_context.geojson", quiet=TRUE)

context_panel_theme <- theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.1, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA))

bottom_guide <- guide_colorbar(direction="horizontal", title.position="bottom", title.hjust=0.5)

p_pov <- ggplot(context_sf) +
  geom_sf(aes(fill=poverty_rate), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=ORANGE, limits=c(pov_ctx_vmin, pov_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE", name="Poverty Rate (%)",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Poverty Rate")

p_unemp <- ggplot(context_sf) +
  geom_sf(aes(fill=unemployment_rate), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=PINK, limits=c(unemp_ctx_vmin, unemp_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE", name="Unemployment Rate (%)",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Unemployment Rate")

p_inc <- ggplot(context_sf) +
  geom_sf(aes(fill=median_household_income), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=GREEN, limits=c(inc_ctx_vmin, inc_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE",
                      labels=function(v) sprintf("$%.0fk", v/1000),
                      name="Median Household Income",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Median Household Income")

p_rpp <- ggplot(context_sf) +
  geom_sf(aes(fill=rpp), color="white", linewidth=0.05) +
  scale_fill_gradient(low="#FCFCFB", high=BLUE, limits=c(rpp_ctx_vmin, rpp_ctx_vmax),
                      oob=scales::squish, na.value="#EEEEEE",
                      name="Regional Price Parity",
                      guide=bottom_guide) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  context_panel_theme +
  labs(title="Regional Price Parity")

(p_pov | p_unemp) / (p_inc | p_rpp) +
  plot_annotation(title="Four Views of County Economic Conditions",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

The geography used to measure these conditions also matters. Much of the previous literature examines broader labor market areas that group several counties together. That scale suits the study of employment adjustment, but it can combine counties with different local prices into a single market. Housing and other costs vary substantially between a metropolitan core and its surrounding counties. A county level design preserves that variation.

A final consideration is how the relationship between task groups and purchasing power is represented. A fixed dollar specification assumes the same shift in task groups corresponds to the same dollar difference everywhere. A proportional one instead implies a larger dollar difference where purchasing power is already high and a smaller one where it is low. The analysis evaluates which form fits rather than assuming either from the outset.

Together these considerations define the gap this study addresses. Prior research shows why task groups matter for local labor markets, and conventional measures describe other dimensions of county conditions. Neither connects the two at the county level in what income can actually buy.

# Data

## Data sources and study period

This study integrated five federal datasets published by three agencies, summarized in <a href="#tbl-sources" class="quarto-xref">Table 1</a>, along with the number of records and variables provided by each source. Most sources reported data at the county level, and they were joined using the county Federal Information Processing Standards (FIPS) code and the year. Regional Price Parities (RPP) were published at the metropolitan area level and required an additional matching step. Counties were first linked to their metropolitan areas using the Census Core Based Statistical Area (CBSA) delineation file and then matched to the corresponding Bureau of Economic Analysis (BEA) price measure ([U.S. Bureau of Economic Analysis 2024](#ref-bea_rpp); [U.S. Census Bureau 2023a](#ref-census_cbsa)).

In [6]:
import pandas as pd
from great_tables import GT

sources_df=pd.DataFrame({
    "Data Source": ["Small Area Income and Poverty Estimates<br>**Census SAIPE**",
                    "Bureau of Economic Analysis Regional Price Parities<br>**BEA RPP**",
                    "Core Based Statistical Area delineation<br>**Census CBSA**",
                    "American Community Survey 1 year estimates<br>**Census ACS**",
                    "Bureau of Labor Statistics Local Area Unemployment Statistics<br>**BLS LAUS**"],
    "Records": [50283, 884, 387, 15690, 88004],
    "Measures": ["median household income, poverty rate",
                 "county price level relative to the national average",
                 "county to metropolitan area crosswalk",
                 "population, occupational employment by category",
                 "unemployment rate"],
    "Destination Table": ["`county_baseline`","`cbsa_rpp`","`cbsa`",
                          "`county_baseline`, `county_task_exposure`","`county_baseline`"],
})

style_table(GT(sources_df)
  .fmt_integer(columns="Records", use_seps=True)
  .fmt_markdown(columns=["Data Source","Destination Table"])
  .cols_align(align="center", columns=["Data Source","Records","Measures","Destination Table"])
  .tab_source_note("Scale: 3,143 counties per year, 15 years (2008 to 2023, excluding 2020), 47,140 county year universe, 11,983 rows in the analytical panel.")
  .tab_source_note("Records count rows retrieved from each source rather than analytical observations. The ACS publishes 1 year estimates only for areas above 65,000 residents, which reduces the universe to the 848 county panel.")
  .tab_source_note("All sources retrieved June 2026 through agency APIs or bulk file download."))

The study spans 2008 through 2023 but excludes 2020, because the American Community Survey (ACS) did not publish data due to disruptions of data collection during the COVID-19 pandemic ([U.S. Census Bureau 2024a](#ref-census_acs)). Those estimates serve as the occupational employment counts utilized to construct the task groups. Over the subsequent fifteen years, the Small Area Income and Poverty Estimates (SAIPE) program will provide an initial sample of 47,140 county-year observations ([U.S. Census Bureau 2024b](#ref-census_saipe)).[1]

## Analytical sample

The initial sample restriction stemmed from the coverage of the American Community Survey occupational estimates. The Census Bureau published ACS 1-year occupational estimates exclusively for counties with populations of at least 65,000. Consequently, counties below this threshold lacked the employment counts necessary to construct the four task groups. Consequently, applying this restriction reduced the sample to 12,087 county-year observations across 848 counties. A subsequent restriction mandated the inclusion of complete model covariates, resulting in the removal of 104 observations with missing unemployment rates.[2] However, population, median household income, and poverty rate data were complete throughout the sample.

The final analytical sample comprised 11,983 county-year observations from 840 counties. All models presented in <a href="#sec-analysis" class="quarto-xref">Section 4</a> and <a href="#sec-results" class="quarto-xref">Section 5</a> utilized these same observations. To ensure unbiased machine learning evaluation, counties were exclusively included in either the training or test set, preventing observations from the same county from appearing in both partitions. <a href="#sec-analysis" class="quarto-xref">Section 4</a> provides a detailed description of the evaluation procedure.

The population threshold imposed a limitation on the scope of the results. While the 840 retained counties accounted for approximately 84 percent of the United States population, they represented a minority of all counties. Consequently, rural counties were underrepresented because the retained counties were generally more populous and metropolitan. It’s important to note that the results were applicable to counties above the American Community Survey publication threshold rather than the entire nation, a limitation discussed in <a href="#sec-conclusions" class="quarto-xref">Section 6</a>.

## Construction of task group measures

Following the framework introduced in <a href="#sec-background" class="quarto-xref">Section 2</a>, we classified broad occupational categories in the American Community Survey 1-year estimates into four task groups using Autor and Dorn ([2013](#ref-AutorDorn2013)). For instance, clerical and sales occupations were categorized as routine cognitive tasks, while service occupations were classified as non-routine manual tasks. Subsequently, we summed employment within each task group for every county and year.

<span id="eq-totals">$$
T_{g,c,t} = \sum_{o \in g} E_{o,c,t}
 \qquad(1)$$</span>

Here, $E_{o,c,t}$ represented employment in occupational category $o$ for county $c$ in year $t$, and $g$ identified the task group. Since the calculation used employment counts without task intensity weights, each value represented the number of jobs in that group. Dividing each group’s total by the total employment, as shown in <a href="#eq-group" class="quarto-xref">Equation 2</a>, converted these values into proportions between zero and one that summed to one.

<span id="eq-group">$$
G_{g,c,t} = \frac{T_{g,c,t}}{\sum_{g'} T_{g',c,t}}
 \qquad(2)$$</span>

These proportions assigned counties of varying sizes to a common scale. Since the four proportions added up to one, non-routine cognitive work was excluded from the regression as the reference group, and the remaining coefficients were interpreted in relation to it. We maintained the four task groups as separate entities because a single routine intensity index would obscure differences between types of work. For instance, a shift away from clerical work and a shift away from assembly work could otherwise appear identical. <a href="#sec-analysis" class="quarto-xref">Section 4</a> provides an explanation of how the task groups were incorporated into the models.

Additionally, the Census Bureau revised its occupational classification between 2009 and 2010. While some category boundaries changed, their assignment to the four task groups remained consistent. <a href="#sec-analysis" class="quarto-xref">Section 4</a> reports the corresponding robustness checks.

## Construction of the purchasing power measure

The outcome was measured as purchasing power to account for variations in what household income could purchase across different counties. We calculated it by adjusting the county median household income from the Census Small Area Income and Poverty Estimates ([U.S. Census Bureau 2024b](#ref-census_saipe)) using Bureau of Economic Analysis Regional Price Parities (RPPs). These RPPs provided local price levels relative to the national average ([U.S. Bureau of Economic Analysis 2024](#ref-bea_rpp)). Since the index was expressed as a percentage, dividing it by 100 converted it into a price multiplier. Subsequently, we divided the median household income by this multiplier to determine purchasing power, as illustrated in <a href="#eq-afford" class="quarto-xref">Equation 3</a>.

<span id="eq-afford">$$
A_{c,t} = \frac{\text{Median household income}_{c,t}}{\text{RPP}_{c,t} / 100}
 \qquad(3)$$</span>

Here, $A_{c,t}$ represented purchasing power for county $c$ in year $t$, and $\text{RPP}_{c,t}$ represented its Regional Price Parity.

For instance, consider a county with a median household income of \$60,000 and a Regional Price Parity of 120. This means that its income could buy approximately what \$50,000 would buy at the national average prices. This adjustment made household income more comparable across counties by accounting for variations in local prices. However, since the Bureau of Economic Analysis didn’t publish separate Regional Price Parities for counties outside metropolitan areas, those counties were assigned the corresponding state-level parity. Within the analytical panel, 45.5 percent of county-year observations used a metropolitan parity, while 54.5 percent used the state measure. Consequently, more than half of the panel relied on a state average that didn’t account for price differences within the state.

### Inflation and year indicators

Regional Price Parities were adjusted for variations across regions but not for changes in the national price level over time. Consequently, purchasing power was reported in nominal U.S. dollars. A national deflator would apply the same annual adjustment to every county. In the log specification, this adjustment would be incorporated into the year indicators, leaving the task group and control coefficients unchanged. Additionally, the year indicators captured other changes that occurred simultaneously across counties within a year, making it impossible to interpret their coefficients solely as inflation. <a href="#sec-analysis" class="quarto-xref">Section 4</a> provides an explanation of how the year indicators were incorporated into the model, while <a href="#sec-conclusions" class="quarto-xref">Section 6</a> addresses this limitation.

## Control variables

The analysis incorporated poverty rate, unemployment rate, and population as county-level covariates. Poverty and unemployment captured the economic distress mentioned in <a href="#sec-background" class="quarto-xref">Section 2</a>, enabling the analysis to determine if task groups provided information beyond these measures. Median household income directly entered the purchasing power outcome, so it was not included as a control variable. Population accounted for variations in county size, which could otherwise affect the estimates.

<a href="#fig-baseline-dists" class="quarto-xref">Figure 3</a> displayed the distributions of poverty rate, unemployment rate, population, and median household income across the same task group panel used in <a href="#fig-dists" class="quarto-xref">Figure 9</a>. All four variables were right skewed, but population was the most extreme and therefore entered the analysis in logarithmic form. Poverty rate, unemployment rate, and median household income remained in their original units. This supported the log respecification in <a href="#sec-analysis" class="quarto-xref">Section 4</a> as a response to the outcome’s own distribution rather than a transformation applied across all variables.

[1] The District of Columbia and Kalawao County, Hawaii, are excluded because they are absent from the county reference file. Connecticut counties leave the panel after 2021 because the state replaced its legacy counties with planning regions beginning in 2022.

[2] All 104 missing observations came from Connecticut’s eight legacy counties, where unemployment rates were unavailable at the required geography ([U.S. Bureau of Labor Statistics 2024](#ref-bls_laus)). Because the missingness reflected geography rather than unemployment itself, it was treated as missing at random.

In [7]:
import os
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()
baseline_engine=create_engine(os.environ["AUTORACK_URL"], pool_pre_ping=True, pool_recycle=300)
baseline_df=pd.read_sql("""
select ca.county_fips, ca.year, cb.poverty_rate, cb.unemployment_rate,
    cb.population, cb.median_household_income
from county_affordability ca
join county_task_exposure cte on ca.county_fips=cte.county_fips and ca.year=cte.year
join county_baseline cb on ca.county_fips=cb.county_fips and ca.year=cb.year
""", baseline_engine)

baseline_names={"poverty_rate":"Poverty Rate (%)", "unemployment_rate":"Unemployment Rate (%)",
                "population":"Population", "median_household_income":"Median Household Income ($)"}
baseline_long=baseline_df.melt(value_vars=list(baseline_names.keys()),
                               var_name="variable", value_name="value")
baseline_long["variable"]=baseline_long["variable"].map(baseline_names)

In [8]:
%%R -i baseline_long -w 9 -h 7 -u in -r 150 -b transparent
baseline_order <- c("Poverty Rate (%)","Unemployment Rate (%)","Population","Median Household Income ($)")
baseline_long$variable <- factor(baseline_long$variable, levels=baseline_order)

p_pov <- ggplot(subset(baseline_long, variable=="Poverty Rate (%)"), aes(x=variable, y=value)) +
  geom_violin(fill=ORANGE, color=ORANGE, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=function(v) sprintf("%g%%", v), breaks=scales::pretty_breaks(n=4)) +
  labs(title="Poverty Rate", x=NULL, y="Rate") +
  theme(axis.text.x=element_blank(), plot.title=element_text(size=11),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank())

p_unemp <- ggplot(subset(baseline_long, variable=="Unemployment Rate (%)"), aes(x=variable, y=value)) +
  geom_violin(fill=PINK, color=PINK, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=function(v) sprintf("%g%%", v), breaks=scales::pretty_breaks(n=4)) +
  labs(title="Unemployment Rate", x=NULL, y="Rate") +
  theme(axis.text.x=element_blank(), plot.title=element_text(size=11),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank())

p_pop <- ggplot(subset(baseline_long, variable=="Population"), aes(x=variable, y=value)) +
  geom_violin(fill=GREY, color=GREY, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_log10(labels=label_comma(), breaks=scales::log_breaks(n=4)) +
  labs(title="Population", x=NULL, y="Population") +
  theme(axis.text.x=element_blank(), plot.title=element_text(size=11),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank())

p_inc <- ggplot(subset(baseline_long, variable=="Median Household Income ($)"), aes(x=variable, y=value)) +
  geom_violin(fill=GREEN, color=GREEN, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=dollar_axis, breaks=scales::pretty_breaks(n=4)) +
  labs(title="Median Household Income", x=NULL, y="Income") +
  theme(axis.text.x=element_blank(), plot.title=element_text(size=11),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank())

(p_pov | p_unemp) / (p_pop | p_inc) +
  plot_annotation(title="Distribution of the Control Variables",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1: Removed 104 rows containing non-finite outside the scale range
(`stat_ydensity()`). 
  
R callback write-console: 2: Removed 104 rows containing non-finite outside the scale range
(`stat_boxplot()`). 
  

## Data architecture and organization

The data architecture separated raw source records from the tables used for analysis. Raw retrievals and intermediate results were stored in a PostgreSQL data lake containing 41 tables, which remained unnormalized to maintain traceability. Processed data were then written to an analytical warehouse containing eleven tables. In this warehouse, county identifiers were standardized, records were restricted to the study period, jurisdictions outside the panel were removed, and foreign key constraints were enforced.

<a href="#fig-erd" class="quarto-xref">Figure 4</a> provided a diagram that summarized the path from federal sources through the data lake and warehouse to the analysis-ready tables.

In [9]:
import graphviz

def entity(name, rows):
    body = f'<TR><TD COLSPAN="3" BGCOLOR="#2A78D6"><FONT COLOR="white" POINT-SIZE="15"><B>{name}</B></FONT></TD></TR>'
    for typ, col, marker in rows:
        m = f'<FONT COLOR="#8F8D87" POINT-SIZE="10">{marker}</FONT>' if marker else ""
        body += f'<TR><TD ALIGN="LEFT"><FONT POINT-SIZE="10">{typ}</FONT></TD><TD ALIGN="LEFT"><FONT POINT-SIZE="13">{col}</FONT></TD><TD ALIGN="LEFT">{m}</TD></TR>'
    return f'<<TABLE BORDER="1" CELLBORDER="0" CELLSPACING="0" CELLPADDING="3">{body}</TABLE>>'

CENSUS_FILL = "#DCEEF7"; CENSUS_BORDER = "#2A78D6"
BEA_FILL = "#DCF3EA"; BEA_BORDER = "#1BAF7A"
BLS_FILL = "#F7DCE6"; BLS_BORDER = "#C2255C"
OUT_FILL = "#FBE3D3"; OUT_BORDER = "#EB6834"
LAKE_FILL = "#EDEDEA"; LAKE_BORDER = "#8F8D87"

dot = f'''
digraph architecture {{
  rankdir=TB
  compound=true
  bgcolor="transparent"
  fontname="Helvetica"
  label=<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="0"><TR><TD><FONT FACE="Helvetica-Bold" POINT-SIZE="24">Data Pipeline Architecture</FONT></TD></TR></TABLE>>
  labelloc="t"
  nodesep=0.4
  ranksep=0.9
  node [fontname="Helvetica", shape=box, style="rounded", color="#C3C2B7"]
  edge [fontname="Helvetica", color="black", arrowsize=0.6]

  subgraph cluster_source {{
    label="Data Sources"
    labeljust="c"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F2F8FC"
    margin=18
    node [style="filled,rounded", margin="0.1,0.05", fontsize=14]
    src_census [label="Census Bureau\\nSAIPE · ACS 1yr · CBSA delineation", fillcolor="{CENSUS_FILL}", color="{CENSUS_BORDER}"]
    src_bea [label="BEA\\nRegional Price Parities", fillcolor="{BEA_FILL}", color="{BEA_BORDER}"]
    src_bls [label="BLS\\nLAUS", fillcolor="{BLS_FILL}", color="{BLS_BORDER}"]
  }}

  subgraph cluster_lake {{
    label="Data Lake"
    labeljust="c"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F4F4F1"
    margin=18
    node [style="filled,rounded", fillcolor="{LAKE_FILL}", color="{LAKE_BORDER}", margin="0.1,0.05", fontsize=14]
    lake [label="Raw retrievals and intermediate results\\n41 tables, unnormalized"]
  }}

  subgraph cluster_warehouse {{
    label="Data Warehouse (3NF)"
    labeljust="c"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#F2FAF6"
    margin=18
    node [shape=plain]

    state [label={entity("state", [("varchar","state_code","PK"),("varchar","state_name","")])}]
    county [label={entity("county", [("bigint","county_fips","PK"),("varchar","county_name",""),("varchar","state_code","FK")])}]
    county_baseline [label={entity("county_baseline", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","population",""),("numeric","median_household_income",""),("numeric","poverty_rate",""),("numeric","unemployment_rate","")])}]
    county_task_exposure [label={entity("county_task_exposure", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","routine_cognitive_share","generated"),("numeric","routine_manual_share","generated"),("numeric","non_routine_cognitive_share","generated"),("numeric","non_routine_manual_share","generated")])}]
    county_affordability [label={entity("county_affordability", [("bigint","county_fips","PK,FK"),("int","year","PK"),("numeric","affordability_salary","")])}]
    county_cbsa_code [label={entity("county_cbsa_code", [("bigint","county_fips","PK,FK"),("varchar","cbsa_code","PK,FK")])}]
    cbsa [label={entity("cbsa", [("varchar","cbsa_code","PK"),("varchar","cbsa_name","")])}]
    cbsa_rpp [label={entity("cbsa_rpp", [("varchar","cbsa_code","PK,FK"),("int","year","PK"),("numeric","rpp_value","")])}]

    {{rank=same; county_baseline; county_task_exposure; county_affordability}}

    state -> county
    county -> county_baseline
    county -> county_task_exposure
    county -> county_affordability
    county -> county_cbsa_code
    county_cbsa_code -> cbsa
    cbsa -> cbsa_rpp
  }}

  subgraph cluster_output {{
    label="Modeling Tables"
    labeljust="c"
    fontsize=16
    fontname="Helvetica-Bold"
    style="rounded"
    color="#C3C2B7"
    bgcolor="#FDF5EF"
    margin=18
    node [style="filled,rounded", fillcolor="{OUT_FILL}", color="{OUT_BORDER}", margin="0.1,0.05", fontsize=14]
    analysis_table [label="Analysis ready county year table\\n(joined on county_fips and year)"]
    models [label="Statistical and ML models"]
    figures [label="Figures and tables"]
    {{rank=same; models; figures}}
    analysis_table -> models
    analysis_table -> figures
  }}

  src_census -> lake [ltail="cluster_source", lhead="cluster_lake", minlen=2]
  src_bea -> lake [ltail="cluster_source", lhead="cluster_lake", minlen=2]
  src_bls -> lake [ltail="cluster_source", lhead="cluster_lake", minlen=2]
  lake -> state [ltail="cluster_lake", lhead="cluster_warehouse", minlen=2]
  county_affordability -> analysis_table [ltail="cluster_warehouse", lhead="cluster_output", minlen=2]
  county_baseline -> analysis_table [style=invis]
  county_task_exposure -> analysis_table [style=invis]
}}
'''
graphviz.Source(dot)

The warehouse is organized around a county table, which is keyed by FIPS code and linked to the state as a reference table. Three analytical tables join to each county by FIPS code and year. The county_baseline table contains population, median household income, poverty rate, and unemployment rate. The county_task_exposure table contains the four task group totals defined in <a href="#eq-totals" class="quarto-xref">Equation 1</a>, while the county_affordability table contains the purchasing power outcome.

# Analysis

In [10]:
import os
import pandas as pd
import numpy as np
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sqlalchemy import create_engine
from dotenv import load_dotenv

load_dotenv()

engine=create_engine(os.environ["AUTORACK_URL"], pool_pre_ping=True, pool_recycle=300)
df=pd.read_sql("""
select ca.county_fips, ca.year, ca.affordability_salary,
    cte.routine_cognitive_share, cte.routine_manual_share,
    cte.non_routine_cognitive_share, cte.non_routine_manual_share,
    cb.poverty_rate, cb.unemployment_rate, cb.population
from county_affordability ca
join county_task_exposure cte on ca.county_fips=cte.county_fips and ca.year=cte.year
join county_baseline cb on ca.county_fips=cb.county_fips and ca.year=cb.year
""", engine)
df["log_population"]=np.log(df["population"])

In [11]:
# level model: OLS with year indicators, standard errors clustered by county
reg_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
          "poverty_rate","unemployment_rate","log_population"]

reg=df.dropna(subset=reg_cols+["affordability_salary"]).copy()

X_panel=pd.concat([reg[reg_cols],
                   pd.get_dummies(reg["year"], prefix="year", drop_first=True).astype(float)], axis=1)
X_panel=sm.add_constant(X_panel)

model=sm.OLS(reg["affordability_salary"], X_panel).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})

reg["fitted"]=model.fittedvalues
reg["resid"]=model.resid

# log respecification, same design matrix
reg["actual_log"]=np.log(reg["affordability_salary"])
model_log=sm.OLS(reg["actual_log"], X_panel).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})
reg["resid_log"]=model_log.resid
reg["fitted_log"]=model_log.fittedvalues

## The unadjusted relationship

The analysis commenced by examining the unadjusted relationship between task groups and purchasing power. <a href="#fig-univariate" class="quarto-xref">Figure 5</a> plotted purchasing power against each of the four task groups across the panel. The two manual groups exhibited the most pronounced negative relationships, while non-routine cognitive tasks showed a positive correlation, and routine cognitive tasks remained relatively flat. None of these relationships were strong enough to stand alone, prompting the subsequent adjustment analysis. Interestingly, the same ordering persisted after the adjustment.

In [12]:
uni_labels={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_cognitive_share":"Non-Routine Cognitive",
    "non_routine_manual_share":"Non-Routine Manual",
}
uni_pts=df.melt(id_vars="affordability_salary", value_vars=list(uni_labels.keys()),
                var_name="group", value_name="share").dropna()
uni_pts["group"]=uni_pts["group"].map(uni_labels)
uni_pts["share"]=uni_pts["share"]*100  # express as percent of county employment

uni_pts["bin"]=uni_pts.groupby("group")["share"].transform(
    lambda s: pd.qcut(s, 20, labels=False, duplicates="drop"))
uni_bins=(uni_pts.groupby(["group","bin"])[["share","affordability_salary"]]
          .mean().reset_index())
uni_order=list(uni_labels.values())

In [13]:
%%R -i uni_pts -i uni_bins -i uni_order -w 10 -h 6 -u in -r 150 -b transparent
uni_pts$group <- factor(uni_pts$group, levels=unlist(uni_order))
uni_bins$group <- factor(uni_bins$group, levels=unlist(uni_order))

# the strip carries the group name alone, centered; the letter tag sits just above
# the top of each panel's y axis, clear of both the strip and the axis labels
tag_df <- data.frame(group=factor(unlist(uni_order), levels=unlist(uni_order)),
                     tag=paste0(LETTERS[seq_along(unlist(uni_order))], ")"))

ggplot(uni_pts, aes(x=share, y=affordability_salary)) +
  geom_point(alpha=0.08, size=0.25, color="grey40") +
  geom_line(data=uni_bins, aes(color=group), linewidth=0.8, show.legend=FALSE) +
  geom_point(data=uni_bins, aes(color=group), size=1.8, show.legend=FALSE) +
  geom_text(data=tag_df, aes(x=-Inf, y=Inf, label=tag), inherit.aes=FALSE,
            hjust=1.2, vjust=-0.45, fontface="bold", size=4.2, color="#0B0B0B") +
  # axes="all_y" repeats the y axis on the right column too; the scale stays shared
  facet_wrap(~group, ncol=2, scales="free_x", axes="all_y") +
  coord_cartesian(clip="off") +
  scale_color_manual(values=TASK_COLORS) +
  scale_x_continuous(breaks=scales::pretty_breaks(n=8)) +
  scale_y_continuous(labels=dollar_axis) +
  labs(title="Purchasing Power Against Each Task Group",
       x="Percent of County Employment", y="Purchasing Power") +
  theme(plot.title=element_text(hjust=0.5),
        strip.text=element_text(size=12, hjust=0.5),
        axis.ticks.x=element_line(color="#C3C2B7", linewidth=0.3),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank(),
        panel.spacing=unit(1.4, "lines"))

## Variance decomposition

Purchasing power and task groups exhibit variations both across counties and within a county over time, each carrying distinct implications. <a href="#fig-between-within" class="quarto-xref">Figure 6</a> effectively divides each group’s variation into these two components. The manual groups show almost complete variation between counties, which contributes to the durability of the purchasing power differences they represent. In contrast, routine cognitive is the only group with substantial within-county movement.

In [14]:
df10=df[df["year"]>=2010].copy()
input_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share"]

within_vals=[]
for col in input_cols:
    yd=df10[col]-df10.groupby("year")[col].transform("mean")
    within=(yd-yd.groupby(df10["county_fips"]).transform("mean")).var()
    within_vals.append(within/yd.var()*100)

bw_labels=["Routine Cognitive","Routine Manual","Non-Routine Manual"]
bw_df=pd.concat([
    pd.DataFrame({"group":bw_labels, "value":[100-v for v in within_vals],
                  "component":"Between Counties"}),
    pd.DataFrame({"group":bw_labels, "value":within_vals,
                  "component":"Within Counties Over Time"}),
], ignore_index=True)
bw_text_df=pd.concat([
    pd.DataFrame({"group":bw_labels, "x":[(100-v)/2 for v in within_vals],
                  "label":[f"{100-v:.0f}%" for v in within_vals], "component":"Between Counties"}),
    pd.DataFrame({"group":bw_labels, "x":[100-v/2 for v in within_vals],
                  "label":[f"{v:.0f}%" for v in within_vals], "component":"Within Counties Over Time"}),
], ignore_index=True)

In [15]:
%%R -i bw_df -i bw_text_df -w 9 -h 3.5 -u in -r 150 -b transparent
bw_df$group <- factor(bw_df$group,
    levels=rev(c("Routine Cognitive","Routine Manual","Non-Routine Manual")))
bw_df$component <- factor(bw_df$component,
    levels=c("Within Counties Over Time","Between Counties"))
bw_text_df$group <- factor(bw_text_df$group, levels=levels(bw_df$group))

# Orange and purple rather than green and red: these are two components of one
# variance, neither good nor bad, so a valenced pair would say the wrong thing.
# Both in-bar labels clear the 4.5:1 floor (orange with ink 6.2, purple with white 13.6).
bw_text_df$text_color <- ifelse(bw_text_df$component=="Within Counties Over Time",
                                 "white", "#0B0B0B")

ggplot(bw_df, aes(x=value, y=group, fill=component)) +
  geom_col(width=0.7) +
  geom_text(data=bw_text_df, aes(x=x, y=group, label=label, color=text_color),
            inherit.aes=FALSE, size=3.4) +
  scale_color_identity() +
  scale_fill_manual(values=c("Between Counties"=ORANGE,
                             "Within Counties Over Time"=PURPLE), name=NULL) +
  scale_x_continuous(breaks=seq(0, 100, 10), expand=c(0, 0)) +
  scale_y_discrete(expand=expansion(add=c(0.3, 0.3))) +
  labs(title="Between County vs. Within County Variation by Task Group",
       x="Share of Variance", y=NULL) +
  # expand=c(0,0) runs the bars flush to 100 with no trailing gap; the right plot
  # margin holds the 100 tick label, which would otherwise clip at the panel edge
  theme(legend.position="bottom", plot.margin=margin(10, 16, 8, 10))

The Census occupation coding change between 2009 and 2010, as described in <a href="#sec-data" class="quarto-xref">Section 3</a>, inflates the apparent variation in routine cognitive occupations within counties. This inflation arises from the coding of the source data rather than any inherent property of the counties. The figure is computed on a panel restricted to 2010 onward, with each year’s cross-county mean removed.

## Model specification

Non-routine cognitive tasks served as the reference group because the four task group proportions summed to one. Each remaining coefficient therefore represented the change in purchasing power associated with a shift from non-routine cognitive work to that group, while holding the other variables constant. This made the coefficients interpretable as comparisons between task groups rather than as independent changes in employment. Task group values ranged from zero to one, and coefficients were divided by 100 when reported as percentage point changes.

Year indicators captured the shared conditions across all counties within a specific year, encompassing events like the 2008 recession, its subsequent recovery, and the national price movements described in <a href="#sec-data" class="quarto-xref">Section 3</a>. By including these indicators, we prevented attributing national changes to variations in county task groups. Since the year coefficients absorbed all common changes within a year, they were treated as nuisance parameters rather than evidence of increasing purchasing power. <a href="#fig-year-effects" class="quarto-xref">Figure 7</a> illustrated their pattern across the panel.

In [16]:
# year coefficients from both specifications, converted to their reporting units:
# dollars for the level model, percent for the log model via (exp(b)-1)*100
def _year_frame(m, pct=False):
    co=m.params.filter(like="year_"); ci=m.conf_int().filter(like="year_", axis=0)
    d=pd.DataFrame({"year":[int(s.split("_")[1]) for s in co.index],
                    "estimate":co.values, "lo":ci[0].values, "hi":ci[1].values})
    if pct:
        for c in ["estimate","lo","hi"]: d[c]=(np.exp(d[c])-1)*100
    return d

year_eff_level=_year_frame(model)
year_eff_log=_year_frame(model_log, pct=True)

In [17]:
%%R -i year_eff_level -i year_eff_log -w 8 -h 6.5 -u in -r 150 -b transparent
year_panel <- function(d, ylab, lab_fn) {
  ggplot(d, aes(x=year, y=estimate)) +
    geom_hline(yintercept=0, linetype="dashed", color=GREY, linewidth=0.4) +
    geom_ribbon(aes(ymin=lo, ymax=hi), fill=BLUE, alpha=0.18) +
    geom_line(color=BLUE, linewidth=0.7) +
    geom_point(color=BLUE, size=1.8) +
    scale_y_continuous(labels=lab_fn) +
    # a tick per study year rather than every fourth, scale unchanged
    scale_x_continuous(breaks=scales::pretty_breaks(n=8)) +
    labs(x=NULL, y=ylab)
}

p_lvl <- year_panel(year_eff_level, "Difference in Purchasing Power", dollar_axis) +
  labs(title="Level Specification")
p_log <- year_panel(year_eff_log, "Percent Difference", function(v) sprintf("%g%%", v)) +
  labs(title="Log Specification")

(p_lvl / p_log) +
  plot_annotation(title="Year Effects Relative to 2008",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

Standard errors were clustered by county because repeated observations within the same county over fifteen years were not independent. A Durbin Watson statistic of 0.496 indicated this dependence. Clustering allowed observations within a county to be correlated, which prevented the uncertainty around the estimated associations from being underestimated. The estimated coefficients are reported in <a href="#sec-results" class="quarto-xref">Section 5</a>.

### Collinearity and the reference group

The unnormalized employment totals of <a href="#eq-totals" class="quarto-xref">Equation 1</a> are not suitable for regression analysis. A variance inflation factor (VIF) quantifies the extent to which a coefficient is affected by its correlation with other inputs, and a value exceeding 10 is considered a serious concern. The four employment totals exhibit factors ranging from 13 to 27, as they all represent employment counts that are proportional to county size and therefore exhibit a strong correlation. When fitted on these totals, the coefficients were unstable, and their standard errors were notably large compared to their magnitudes, which are typical indicators of multicollinearity. The specification proposed by <a href="#fig-vif" class="quarto-xref">Figure 8</a> addresses this issue twice. Normalizing the data by the four group total, as described in <a href="#sec-data" class="quarto-xref">Section 3</a>, eliminates the shared scale, while dropping non-routine cognitive as the reference group removes the constraint that the four proportions must sum to one. Consequently, every factor in the estimated specification falls within the range of 1.2 to 1.6.

In [18]:
raw=pd.read_sql("""
    select routine_cognitive, routine_manual, non_routine_cognitive, non_routine_manual
    from county_task_exposure
    """, engine)

raw_cols=["routine_cognitive","routine_manual","non_routine_cognitive","non_routine_manual"]
X_raw=sm.add_constant(raw[raw_cols])
vif_raw=pd.Series([variance_inflation_factor(X_raw.values, i) for i in range(1, X_raw.shape[1])],
                  index=raw_cols)

X_vif_level=sm.add_constant(reg[reg_cols])
vif_share=pd.Series([variance_inflation_factor(X_vif_level.values, i) for i in range(1, X_vif_level.shape[1])],
                    index=reg_cols)

group_display=["Routine Cognitive","Routine Manual","Non-Routine Cognitive","Non-Routine Manual"]

vif_dot_df=pd.concat([
    pd.DataFrame({"group":group_display, "vif":vif_raw.values,
                  "spec":"Unnormalized Employment Totals"}),
    pd.DataFrame({"group":["Routine Cognitive","Routine Manual","Non-Routine Manual"],
                  "vif":vif_share[["routine_cognitive_share","routine_manual_share",
                                   "non_routine_manual_share"]].values,
                  "spec":"Group Proportions, as Estimated"}),
], ignore_index=True)

# wide form for the before-to-after arrow on each row (excludes the reference group,
# which has no estimated point)
vif_seg_df=pd.DataFrame({
    "group":["Routine Cognitive","Routine Manual","Non-Routine Manual"],
    "raw":vif_raw[["routine_cognitive","routine_manual","non_routine_manual"]].values,
    "share":vif_share[["routine_cognitive_share","routine_manual_share",
                       "non_routine_manual_share"]].values,
})

In [19]:
%%R -i vif_dot_df -w 8 -h 4 -u in -r 150 -b transparent
vif_dot_df$group <- factor(vif_dot_df$group,
    levels=rev(c("Routine Cognitive","Routine Manual",
                 "Non-Routine Cognitive","Non-Routine Manual")))
vif_dot_df$spec <- factor(vif_dot_df$spec,
    levels=c("Group Proportions, as Estimated","Unnormalized Employment Totals"),
    labels=c("Group proportions","Employment counts"))

ggplot(vif_dot_df, aes(x=vif, y=group)) +
  # bold black thresholds, heavier than the connector segments so they read as
  # reference lines rather than as part of the data
  geom_vline(xintercept=5, linetype="dotted", linewidth=1.1, color="#0B0B0B") +
  geom_vline(xintercept=10, linetype="dashed", linewidth=1.2, color="#0B0B0B") +
  annotate("text", x=5, y=Inf, label="VIF = 5", hjust=-0.15, vjust=1.4, color="#0B0B0B", size=3.1, fontface="bold") +
  annotate("text", x=10, y=Inf, label="VIF = 10", hjust=-0.15, vjust=1.4, color="#0B0B0B", size=3.3, fontface="bold") +
  geom_line(aes(group=group), color="#0B0B0B", linewidth=0.6) +
  geom_point(aes(color=spec), size=3.2) +
  geom_text(aes(label=sprintf("%.1f", vif), color=spec), vjust=-1.1, size=3.2, show.legend=FALSE) +
  scale_x_continuous(limits=c(0, 30), breaks=seq(0, 30, by=2.5)) +
  scale_y_discrete(labels=function(x) ifelse(x=="Non-Routine Cognitive", "Non-Routine Cognitive *", x)) +
  scale_color_manual(values=c("Group proportions"=GREEN,
                              "Employment counts"=ORANGE)) +
  labs(title="Variance Inflation Factors by Task Specification",
       x="Variance Inflation Factor", y=NULL, color=NULL) +
  theme(legend.position="bottom",
        plot.title=element_text(face="bold", hjust=0.5),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank(),
        axis.ticks.x=element_line(color="#0B0B0B", linewidth=0.6),
        axis.ticks.length.x=unit(6, "pt"))

## Specification diagnostics

### Distribution of the analytical variables

The analysis next delved into the distributions of the variables employed in the specification. <a href="#fig-dists" class="quarto-xref">Figure 9</a> presented purchasing power and the four task groups across the panel, emphasizing the skewness and outliers that guided the diagnostic checks that followed. Purchasing power exhibited right skewness, with a skewness coefficient of 1.17 and a central tendency near \$57,000. A long upper tail extended beyond the center of the distribution, making the extremes particularly crucial for assessing model fit.

In [20]:
dist_label_map={
    "affordability_salary":"Purchasing Power ($)",
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_cognitive_share":"Non-Routine Cognitive",
    "non_routine_manual_share":"Non-Routine Manual",
}
dists_df=df.melt(value_vars=list(dist_label_map.keys()),
                 var_name="variable", value_name="value")
dists_df["variable"]=dists_df["variable"].map(dist_label_map)

In [21]:
%%R -i dists_df -w 10 -h 5.5 -u in -r 150 -b transparent
pp_vals <- subset(dists_df, variable=="Purchasing Power ($)")$value
pp_ylim <- c(0, max(pp_vals) * 1.02)

p_pp <- ggplot(subset(dists_df, variable=="Purchasing Power ($)"), aes(x=variable, y=value)) +
  geom_violin(fill=BLUE, color=BLUE, alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", color="grey30", alpha=0.7,
               outlier.size=0.8, outlier.alpha=0.5) +
  scale_y_continuous(labels=dollar_axis, breaks=scales::pretty_breaks(n=7)) +
  coord_cartesian(ylim=pp_ylim) +
  labs(title="Purchasing Power", x=NULL, y="Purchasing Power") +
  theme(axis.text.x=element_blank(), axis.ticks.x=element_blank(),
        plot.margin=margin(3, 6, 3, 6)) +
  theme(plot.title=element_text(size=11))

task_box_df <- subset(dists_df, variable %in% c("Routine Cognitive", "Routine Manual",
                                                 "Non-Routine Cognitive", "Non-Routine Manual"))
task_box_df$variable <- factor(task_box_df$variable,
    levels=c("Non-Routine Cognitive", "Non-Routine Manual", "Routine Cognitive", "Routine Manual"))

p_tasks <- ggplot(task_box_df, aes(x=variable, y=value, fill=variable, color=variable)) +
  geom_violin(alpha=0.35, linewidth=0.6, trim=TRUE) +
  geom_boxplot(width=0.12, fill="white", alpha=0.7,
               outlier.size=0.6, outlier.alpha=0.4) +
  scale_fill_manual(values=TASK_COLORS, guide="none") +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  scale_y_continuous(labels=percent, breaks=scales::pretty_breaks(n=7)) +
  coord_cartesian(ylim=c(0.05, 0.65)) +
  scale_x_discrete(labels=function(x) gsub(" ", "\n", x)) +
  labs(title="Task Groups", x=NULL, y=NULL) +
  theme(axis.text.x=element_text(size=8))

(p_pp | p_tasks) + plot_layout(widths=c(1, 1.6)) +
  plot_annotation(title="Distribution of Purchasing Power and Task Groups",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

### Residual and quantile diagnostics

A linear model assumes a constant rate of association across the data range and symmetric, evenly distributed errors around the fit. Three diagnostics test these assumptions: residuals plotted against fitted values, a quantile-quantile (QQ) plot of the residuals, and the variance inflation factors already reported. The raw scale model shows clear systematic curvature in the residuals and a marked upper tail departure in the QQ plot (<a href="#fig-diagnostics" class="quarto-xref">Figure 12</a>, Panels A and B). The model underestimates both the lowest and highest purchasing power counties while fitting the middle well, and the upper tail departure indicates a residual skew of 1.01.

The level model still provides the interpretable dollar estimates we report. However, its own diagnostics reveal that the data exhibit structure that a straight line in dollars cannot represent. The curvature in the residual smoother and the widening spread with fitted values suggest a specific failure in the fixed slope specification, and each subsequent model relaxes the assumption behind it.

### Influential points

In [22]:
cooks_infl=model.get_influence()
reg["cooks_d"]=cooks_infl.cooks_distance[0]
reg["leverage"]=cooks_infl.hat_matrix_diag
reg["std_resid"]=cooks_infl.resid_studentized_internal

county_state_names=pd.read_sql(
    "select c.county_fips, c.county_name, s.state_name "
    "from county c join state s on c.state_code=s.state_code", engine)

influence_pts=(reg[["county_fips","year","leverage","std_resid","cooks_d"]]
                .merge(county_state_names, on="county_fips", how="left"))
influence_pts["label"]=influence_pts["county_name"]+", "+influence_pts["state_name"]

# Cook's distance contours. D = (r^2 / p) * (h / (1 - h)), so a given D traces
# r = +/- sqrt(D * p * (1 - h) / h) across the leverage range. The conventional
# 0.5 and 1.0 contours sit far outside the data here, so the two drawn levels are
# the ones this analysis actually uses: the 99th percentile cutoff behind the
# sensitivity refit below, and the observed maximum.
n_params=model.df_model+1
d99_level=float(reg["cooks_d"].quantile(0.99))
dmax_level=float(reg["cooks_d"].max())
r_lo, r_hi=float(reg["std_resid"].min()), float(reg["std_resid"].max())
h_grid=np.linspace(reg["leverage"].min(), reg["leverage"].max(), 400)

contour_frames=[]
for level, level_name in [(d99_level, f"Cook's D = {d99_level:.4f} (99th percentile)"),
                          (dmax_level, f"Cook's D = {dmax_level:.4f} (observed maximum)")]:
    r_curve=np.sqrt(level*n_params*(1-h_grid)/h_grid)
    for sign in (1, -1):
        contour_frames.append(pd.DataFrame({"h": h_grid, "r": sign*r_curve,
                                            "level": level_name,
                                            "branch": f"{level_name}{sign}"}))
influence_contours=pd.concat(contour_frames)
influence_contours=influence_contours[(influence_contours["r"]>=r_lo-0.3)
                                      &(influence_contours["r"]<=r_hi+0.3)]

# label each county once, at its own most influential year
influence_labels=(influence_pts.loc[influence_pts.groupby("label")["cooks_d"].idxmax()]
                   .nlargest(4, "cooks_d").copy())
influence_labels["label"]=(influence_labels["label"].str.replace(" County", "", regex=False)
                           +" "+influence_labels["year"].astype(str))
influence_labels["ny"]=[0.0, 0.0, 0.42, -0.42]
r_at_threshold=float(np.sqrt(1.0*n_params*(1-reg["leverage"].max())/reg["leverage"].max()))

# sensitivity check: refit excluding the top 1 percent most influential rows
influence_cutoff=reg["cooks_d"].quantile(0.99)
influence_keep=reg["cooks_d"]<=influence_cutoff
n_excluded=int((~influence_keep).sum())

model_robust=sm.OLS(reg.loc[influence_keep, "affordability_salary"], X_panel.loc[influence_keep]).fit(
    cov_type="cluster", cov_kwds={"groups": reg.loc[influence_keep, "county_fips"]})

sens_labels={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_manual_share":"Non-Routine Manual",
    "poverty_rate":"Poverty Rate",
    "unemployment_rate":"Unemployment Rate",
    "log_population":"Log Population",
}
sensitivity_df=pd.DataFrame({
    "Variable": [sens_labels[c] for c in reg_cols],
    "Full Sample": [model.params[c] for c in reg_cols],
    "Excluding Top 1%": [model_robust.params[c] for c in reg_cols],
})
sensitivity_df["Change (%)"]=(sensitivity_df["Excluding Top 1%"]/sensitivity_df["Full Sample"]-1)*100

A fourth diagnostic, Cook’s distance, evaluates whether any single county year disproportionately influences the level specification’s coefficients. None of the observations exceed the conventional threshold of 1.0, so no point warrants further investigation. The highest value recorded is 0.009. <a href="#fig-cooks-distance" class="quarto-xref">Figure 10</a> plots every county year by its leverage and standardized residual, with dashed curves tracing constant Cook’s distance. The conventional 0.5 and 1.0 contours fall well outside the plotted range, since reaching 1.0 at the highest observed leverage would require a standardized residual near 40. The labeled points are the four counties with the largest values, each shown at its own most influential year. Loudoun and Stafford County, Virginia, are both high purchasing power counties located in the Washington D.C. exurbs. Williamson County, Tennessee, is a high purchasing power Nashville exurb. Apache County, Arizona, is a low purchasing power county situated on the Navajo Nation. These counties are the same high-end outliers that <a href="#fig-dists" class="quarto-xref">Figure 9</a> already depicts in the purchasing power distribution.

In [23]:
%%R -i influence_pts -i influence_contours -i influence_labels -i r_at_threshold -w 9 -h 5.5 -u in -r 150 -b transparent
ggplot() +
  geom_hline(yintercept=0, color=GREY, linewidth=0.4) +
  geom_point(data=influence_pts, aes(x=leverage, y=std_resid),
             size=0.5, color=BLUE, alpha=0.18) +
  geom_line(data=influence_contours, aes(x=h, y=r, group=branch, color=level),
            linetype="dashed", linewidth=0.55) +
  scale_color_manual(values=c(GREY, RED)) +
  geom_point(data=influence_labels, aes(x=leverage, y=std_resid), size=1.9, color=RED) +
  geom_text(data=influence_labels, aes(x=leverage, y=std_resid+ny, label=label),
            size=3.0, color="#252525", hjust=-0.11) +
  annotate("text", x=Inf, y=-Inf, hjust=1.03, vjust=-0.55, size=3.0,
           color="#52514E", lineheight=1.05,
           label=sprintf("n = 11,983    max Cook's D = %.4f\nReaching the conventional 1.0 threshold would take a standardized residual near %.0f",
                         max(influence_pts$cooks_d), r_at_threshold)) +
  scale_x_continuous(labels=scales::label_number(accuracy=0.001),
                     expand=expansion(mult=c(0.02, 0.24))) +
  scale_y_continuous(expand=expansion(mult=c(0.20, 0.07))) +
  guides(color=guide_legend(nrow=1)) +
  labs(title="Residuals Against Leverage With Cook's Distance Contours",
       x="Leverage (Hat Value)", y="Standardized Residual")

/opt/anaconda3/envs/capstone/lib/python3.12/site-packages/rpy2/robjects/pandas2ri.py:56: UserWarning: DataFrame contains duplicated elements in the index, which will lead to loss of the row names in the resulting data.frame
  warnings.warn('DataFrame contains duplicated elements in the index, '

No single point threatened the overall fit, so the more informative check was to refit the model after removing the most influential 1 percent of the panel, 120 county year observations. <a href="#tbl-influence-sensitivity" class="quarto-xref">Table 2</a> reported the results. The task group coefficients changed by single digit to low double digit percentages, while the poverty coefficient remained relatively stable. This indicated that the main associations were not driven by a small number of extreme counties.

Unemployment rate was the exception. As reported in <a href="#sec-results" class="quarto-xref">Section 5</a>, it was the smallest and least precisely estimated control, and its coefficient changed by approximately one third. A second stability check examined whether the main estimates were consistent over time. The model was refit separately for the pre pandemic period from 2010 to 2019 and the post pandemic period from 2021 to 2023, with <a href="#sec-results" class="quarto-xref">Section 5</a> reporting the stability of the coefficients across the two periods.

In [24]:
style_table(GT(sensitivity_df)
  .fmt_currency(columns=["Full Sample","Excluding Top 1%"], decimals=0)
  .fmt_number(columns="Change (%)", decimals=1)
  .tab_source_note(f"{n_excluded} of {len(reg)} county year observations excluded, the top 1% by Cook's distance."))

## Log respecification

The level model indicated that a constant dollar association was insufficient to accurately describe the relationship. Logging purchasing power was employed to determine if the association was better represented proportionally, enabling a comparable shift in a task group to correspond to a larger dollar difference in counties with higher purchasing power. The <a href="#fig-log-transform" class="quarto-xref">Figure 11</a> visualization demonstrated how the transformation mitigated the right skew in purchasing power and laid the groundwork for reevaluating the residual pattern.

In [25]:
level_skew=float(reg["affordability_salary"].skew())
log_skew=float(reg["actual_log"].skew())
logt_df=pd.concat([
    pd.DataFrame({"value":reg["affordability_salary"], "dist":"level"}),
    pd.DataFrame({"value":reg["actual_log"], "dist":"log"}),
], ignore_index=True)

In [26]:
%%R -i logt_df -i level_skew -i log_skew -w 11 -h 4.8 -u in -r 150 -b transparent

lv <- subset(logt_df, dist=="level")$value
lg <- subset(logt_df, dist=="log")$value

# density curve with the median and mean marked; the gap between the two lines is
# the skew each panel reports, made visible rather than only stated
dens_panel <- function(v, title, skew, xlab, dollars, fill) {
  # density on the dollar scale is ~2e-05, which R prints in scientific notation;
  # force plain decimals there and leave the log panel on default labels
  y_labels <- if (dollars) function(v) sprintf("%.5f", v) else waiver()
  d <- density(v)
  dd <- data.frame(x=d$x, y=d$y)
  md <- median(v); mn <- mean(v); ytop <- max(dd$y)
  p <- ggplot(dd, aes(x=x, y=y)) +
    geom_area(fill=fill, alpha=0.9) +
    geom_line(color="#0B0B0B", linewidth=0.7) +
    geom_vline(xintercept=md, color="#0B0B0B", linewidth=0.6) +
    geom_vline(xintercept=mn, color="#0B0B0B", linewidth=0.6, linetype="dashed") +
    annotate("text", x=md, y=ytop*1.06, label="Median", size=2.9, hjust=1.12, color="#0B0B0B") +
    annotate("text", x=mn, y=ytop*1.06, label="Mean", size=2.9, hjust=-0.12, color="#0B0B0B") +
    scale_y_continuous(labels=y_labels, expand=expansion(mult=c(0, 0.12))) +
    labs(title=title, subtitle=sprintf("Skewness = %.2f", skew), x=xlab, y="Density") +
    theme(plot.title=element_text(size=11),
          plot.subtitle=element_text(size=9, hjust=0.5, face="italic"))
  if (dollars) p <- p + scale_x_continuous(labels=dollar_axis)
  p
}

p_level <- dens_panel(lv, "Original Scale", level_skew, "Purchasing Power", TRUE, ORANGE)
p_log   <- dens_panel(lg, "Log Scale", log_skew, "Log Purchasing Power", FALSE, GREEN)

(p_level | p_log) +
  plot_annotation(title="Distribution of Purchasing Power Before and After Log Transformation",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

Refitting the same specification using log purchasing power resolved most of the problems identified in the level model. <a href="#fig-diagnostics" class="quarto-xref">Figure 12</a> showed less systematic curvature in the residuals, while the QQ points followed the reference line more closely after the transformation. Some tail deviation remained, but residual skew declined from 1.01 to 0.09. The log linear specification therefore provided the baseline for evaluating the more flexible models that followed.

In [27]:
resid_df=reg[["fitted","resid"]].copy()
resid_log_df=reg[["fitted_log","resid_log"]].copy()

In [28]:
%%R -i resid_df -i resid_log_df -w 9 -h 7 -u in -r 150 -b transparent
resid_df$std_resid <- as.numeric(scale(resid_df$resid))
resid_log_df$std_resid <- as.numeric(scale(resid_log_df$resid_log))

p1 <- ggplot(resid_df, aes(x=fitted, y=resid)) +
  geom_bin2d(bins=55, aes(fill=after_stat(count))) +
  scale_fill_gradient(low="#E8EEF4", high=BLUE, guide="none") +
  geom_hline(yintercept=0, linewidth=0.4, linetype="dashed", color="grey40") +
  geom_smooth(method="loess", formula=y~x, se=FALSE, color=RED, linewidth=0.9) +
  scale_x_continuous(labels=dollar_axis) +
  scale_y_continuous(labels=dollar_axis, breaks=scales::pretty_breaks(n=7)) +
  labs(x="Fitted Purchasing Power", y="Residual, Raw", title="Residuals vs. Fitted") +
  theme(plot.title=element_text(size=11))

p2 <- ggplot(resid_df, aes(sample=std_resid)) +
  stat_qq(color=BLUE, alpha=0.35, size=0.8) +
  stat_qq_line(color="#0B0B0B", linewidth=0.8) +
  scale_y_continuous(breaks=scales::pretty_breaks(n=7)) +
  labs(x="Theoretical Quantiles", y="Standardized Residual", title="Normal Q-Q") +
  theme(plot.title=element_text(size=11))

p3 <- ggplot(resid_log_df, aes(x=fitted_log, y=resid_log)) +
  geom_bin2d(bins=55, aes(fill=after_stat(count))) +
  scale_fill_gradient(low="#E8F4EE", high=GREEN, guide="none") +
  geom_hline(yintercept=0, linewidth=0.4, linetype="dashed", color="grey40") +
  geom_smooth(method="loess", formula=y~x, se=FALSE, color=RED, linewidth=0.9) +
  scale_y_continuous(breaks=scales::pretty_breaks(n=7)) +
  labs(x="Fitted Log Purchasing Power", y="Residual, Log")

p4 <- ggplot(resid_log_df, aes(sample=std_resid)) +
  stat_qq(color=GREEN, alpha=0.35, size=0.8) +
  stat_qq_line(color="#0B0B0B", linewidth=0.8) +
  scale_y_continuous(breaks=scales::pretty_breaks(n=7)) +
  labs(x="Theoretical Quantiles", y="Standardized Residual")

# horizontal gridlines off on every panel; the dashed zero line and the QQ
# reference line are the references that matter here
grid_off <- theme(panel.grid.major.y=element_blank(), panel.grid.minor.y=element_blank())
p1 <- p1 + grid_off; p2 <- p2 + grid_off; p3 <- p3 + grid_off; p4 <- p4 + grid_off

(p1 | p2) / (p3 | p4) +
  plot_annotation(title="Regression Diagnostics, Level vs. Log Specification",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

<a href="#fig-binned-scale" class="quarto-xref">Figure 13</a> presents the same comparison in the units of the outcome. On the level scale, the model closely tracks purchasing power throughout the middle of the distribution, where most counties are located. However, both tail bins deviate substantially from their predictions, coming in approximately \$8,000 to \$9,000 above the expected values. On the log scale, the points closely follow the line throughout the distribution.

In [29]:
reg.loc[:, "bin"]=pd.qcut(reg["fitted"], 20, labels=False)
binned=reg.groupby("bin")[["fitted","affordability_salary"]].mean()
reg.loc[:, "bin_log"]=pd.qcut(reg["fitted_log"], 20, labels=False)
binned_log=reg.groupby("bin_log")[["fitted_log","actual_log"]].mean()

pts_df=pd.concat([
    pd.DataFrame({"fitted":reg["fitted"], "actual":reg["affordability_salary"],
                  "target":"Level Target"}),
    pd.DataFrame({"fitted":reg["fitted_log"], "actual":reg["actual_log"],
                  "target":"Log Target"}),
], ignore_index=True)
bins_df=pd.concat([
    pd.DataFrame({"fitted":binned["fitted"], "actual":binned["affordability_salary"],
                  "target":"Level Target"}),
    pd.DataFrame({"fitted":binned_log["fitted_log"], "actual":binned_log["actual_log"],
                  "target":"Log Target"}),
], ignore_index=True)

In [30]:
%%R -i pts_df -i bins_df -w 10 -h 5 -u in -r 150 -b transparent
dollar_axis <- function(v) ifelse(is.na(v), "", ifelse(v == 0, "$0",
  sprintf("%s$%s", ifelse(v < 0, "-", ""), formatC(abs(v), format="d", big.mark=","))))

# patchwork rather than facets: the two panels carry different units, so each needs
# its own axis titles, and only separate plots can take A) and B) tags
panel <- function(tg, col, xlab, ylab, title, dollars) {
  p <- ggplot(subset(pts_df, target==tg), aes(x=fitted, y=actual)) +
    geom_point(alpha=0.05, size=0.3, color="grey50") +
    geom_abline(slope=1, intercept=0, linewidth=0.9, color="#0B0B0B") +
    geom_point(data=subset(bins_df, target==tg), size=2.4, color=col) +
    scale_x_continuous(breaks=scales::pretty_breaks(n=5),
                       labels=if (dollars) dollar_axis else waiver()) +
    scale_y_continuous(breaks=scales::pretty_breaks(n=5),
                       labels=if (dollars) dollar_axis else waiver()) +
    labs(x=xlab, y=ylab, title=title) +
    theme(plot.title=element_text(size=11))
  p
}

p_level <- panel("Level Target", ORANGE, "Fitted Purchasing Power",
                 "Actual Purchasing Power", "Level Target", TRUE)
p_log   <- panel("Log Target", GREEN, "Fitted Log Purchasing Power",
                 "Actual Log Purchasing Power", "Log Target", FALSE)

(p_level | p_log) +
  plot_annotation(title="Binned Fit Accuracy, Level vs. Log Specification",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

The residual pattern did not disappear entirely after the transformation. The remaining structure could reflect nonlinearity, interactions among the variables, or omitted factors that the diagnostics alone could not distinguish. This motivated the use of a random forest, which could capture nonlinear relationships and interactions without specifying them in advance. The log linear specification served as the benchmark, allowing us to test whether a more flexible model captured additional signal in held out counties.

## Predictive modeling

The predictive comparison utilized the county year panel assembled earlier, incorporating state identifiers for the subsequent models. Model complexity only increased when additional flexibility enhanced performance on the held-out counties. The log respecification addressed the issues identified in the level model, while the remaining residual structure prompted testing a random forest capable of capturing nonlinear relationships and interactions. A three hidden layer feedforward neural network served as an additional test to determine if greater model flexibility further improved performance.

All models were evaluated on the same held-out counties from the grouped split, so any differences in performance reflected model structure rather than variations in the evaluation observations. If the more flexible models failed to improve held-out performance, the log linear specification was retained as the simpler representation of the relationship.

In [31]:
# state_code isn't in the panel regression's df, add it here
state_lookup=pd.read_sql(
    "select c.county_fips, c.state_code, s.state_name "
    "from county c join state s on c.state_code=s.state_code", engine
)
state_name_map=state_lookup.drop_duplicates("state_code").set_index("state_code")["state_name"]
df_ml=df.merge(state_lookup, on="county_fips", how="left")

### Evaluation design

#### Feature set and reference groups

In [32]:
feature_cols=[
    "routine_cognitive_share",
    "routine_manual_share",
    "non_routine_cognitive_share",
    "non_routine_manual_share",
    "poverty_rate",
    "unemployment_rate",
    "year",
    "state_code",
]

model_df=df_ml.dropna(subset=feature_cols).reset_index(drop=True).copy()

Each task group was defined based on its proportion of total employment, ensuring that the sum of the four values equaled one for each county and year. Since increasing one group inevitably reduced at least one of the others, all four proportions could not simultaneously enter the linear specification without perfect collinearity. Consequently, non-routine cognitive tasks were excluded as the reference group, aligning with the panel regression approach. This decision maintained consistency in interpreting the task group coefficients across both inferential and predictive analyses.

State indicators also required a comparable reference category. <a href="#fig-state-afford" class="quarto-xref">Figure 14</a> illustrated the variation in mean purchasing power across states, identifying a state situated near the center of the distribution as the reference point. By selecting a state near the middle, comparisons were avoided to states with exceptionally high or low purchasing power, simplifying the interpretation of the resulting coefficients. While the choice of reference state did not affect the overall model fit, it influenced the expression of the state coefficients.

#### Grouped train and test split

This is not a forecasting exercise, and a time ordered split is not required for its usual reason. A grouped split is still necessary, because a county’s economic profile barely moves year to year, which makes its 2018 and 2019 rows near duplicates. If two adjacent rows land on opposite sides of the split, the model can recall a county rather than learn a relationship. We group by county and split on row membership before any further feature engineering. The split assigns 9,565 county year observations to training and holds out the remaining 2,418, with no county contributing rows to both. Every test set prediction therefore comes from a county the model has seen nothing of, which tests whether the pattern generalizes to new places.

In [33]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.preprocessing import StandardScaler

target_col="affordability_salary"
groups=model_df["county_fips"]

gss=GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx=next(gss.split(model_df, groups=groups))

groups_train=groups.iloc[train_idx]

assert len(set(groups.iloc[train_idx]) & set(groups.iloc[test_idx]))==0

#### Year and state indicators

A grouped train and test split was employed because observations from the same county were closely related across years. A random split could inadvertently place observations from the same county in both sets, potentially allowing the model to leverage information about a county it had already encountered. By grouping observations by county, we effectively eliminated this overlap and provided a more robust test of whether the observed relationship could generalize to new locations.

The split allocated 9,565 county-year observations to the training set and 2,418 to the testing set, ensuring that each county appeared in only one of the two sets. All feature engineering was conducted after the split to maintain this separation. Consequently, every test observation originated from a county that was not represented in the training data.

In [34]:
model_df["year"]=model_df["year"].astype("category")
model_df["state_code"]=model_df["state_code"].astype("category")

REFERENCE_YEAR=model_df["year"].cat.categories.min()  # earliest year as reference

# pool states with too few counties to support a stable, independent coefficient
county_counts=model_df.groupby("state_code", observed=True)["county_fips"].nunique()
THIN_STATE_THRESHOLD=5
thin_states=county_counts[county_counts<THIN_STATE_THRESHOLD].index.tolist()

model_df["state_code_grouped"]=model_df["state_code"].astype(str)
model_df["state_code_grouped"]=model_df["state_code_grouped"].where(
    ~model_df["state_code_grouped"].isin(thin_states), "OTHER"
)
model_df["state_code_grouped"]=model_df["state_code_grouped"].astype("category")

state_affordability=model_df.groupby("state_code_grouped", observed=True)["affordability_salary"].mean().sort_values()

median_state=state_affordability.index[len(state_affordability)//2]
REFERENCE_STATE=median_state
# resolved once here so the prose, the figure accent, and the held out indicator
# all name the same state rather than each re-deriving it
reference_state_label=state_name_map.get(REFERENCE_STATE, str(REFERENCE_STATE))

year_dummies=pd.get_dummies(model_df["year"], prefix="year")
year_dummies=year_dummies.drop(columns=[f"year_{REFERENCE_YEAR}"])

state_dummies=pd.get_dummies(model_df["state_code_grouped"], prefix="state")
state_dummies=state_dummies.drop(columns=[f"state_{REFERENCE_STATE}"])

model_df=pd.concat([model_df, year_dummies, state_dummies], axis=1)

year_cols_model=list(year_dummies.columns)
state_cols_model=list(state_dummies.columns)

The reference state was Texas, whose mean purchasing power was close to the median of the state distribution depicted in <a href="#fig-state-afford" class="quarto-xref">Figure 14</a>. By selecting a state near the center, we avoided anchoring comparisons to an unusually high or low purchasing power state, making the coefficients more interpretable. While most state coefficients remained within a few percent of the reference, the largest differences reached 17 percent. Once the economic controls and year indicators were included, as demonstrated by the permutation results below, state indicators contributed little to held-out performance.

In [35]:
state_df=state_affordability.reset_index()
state_df.columns=["state","mean_pp"]
state_df["state_label"]=state_df["state"].map(state_name_map)
# the pooled thin-state bucket is not a state; excluded from the ranking display below
state_df=state_df[state_df["state"]!="OTHER"].copy()
state_median=float(state_affordability.median())
state_df["comparison"]=state_df["mean_pp"].ge(state_median).map({True:"Above Average", False:"Below Average"})

In [36]:
%%R -i state_df -i state_median -i reference_state_label -w 8 -h 10 -u in -r 150 -b transparent
state_df$state_label <- factor(state_df$state_label, levels=state_df$state_label)
# accent the state the models hold out as the reference. Matching on the label rather
# than re-deriving the median keeps the figure and the fitted models in agreement;
# %in% leaves every point unaccented if the reference is the pooled bucket
ACCENT <- BLUE
ref_i <- which(state_df$state_label == reference_state_label)
state_df$pt_color <- ifelse(seq_len(nrow(state_df)) %in% ref_i, ACCENT, "#0B0B0B")
state_df$pt_size  <- ifelse(seq_len(nrow(state_df)) %in% ref_i, 3.8, 2.6)
median_label <- sprintf("Median: $%.0fk", state_median/1000)
n_states <- length(levels(state_df$state_label))
pp_range <- range(state_df$mean_pp)

# axis starts at $30k rather than zero. The Lie Factor concern that motivated a
# zero anchor does not bind here: each segment runs from the median line to the
# state's value, so its length encodes deviation from the median rather than
# absolute magnitude, and moving the left edge rescales every segment equally.
x_lo <- 30000
x_hi <- pp_range[2] * 1.05

# round $20k breaks. Offsets from the median were tried and reverted: anchoring
# every tick to the median produced labels like $4.47k and $9.47k that collided.
# The median keeps its own dotted line and its labeled annotation instead.
state_breaks <- seq(0, 200000, by=10000)
state_breaks <- state_breaks[state_breaks >= x_lo & state_breaks <= x_hi]

# annotation centered within each visual half of the zero-anchored panel
low_mid  <- (x_lo + state_median) / 2
high_mid <- (state_median + x_hi) / 2

ggplot(state_df, aes(x=mean_pp, y=state_label)) +
  annotate("rect", xmin=-Inf, xmax=state_median, ymin=-Inf, ymax=Inf,
           fill="#C0392B", alpha=0.10) +
  annotate("rect", xmin=state_median, xmax=Inf, ymin=-Inf, ymax=Inf,
           fill=GREEN, alpha=0.10) +
  # wrapped onto two lines so neither label crosses the median line or runs off
  # the panel edge, which the single line versions did at size 6
  annotate("text", x=low_mid, y=n_states / 2, label="Lower\nPurchasing\nPower",
           color="#C0392B", fontface="bold", size=5.5, hjust=0.5, vjust=0.5,
           alpha=1, lineheight=0.95) +
  annotate("text", x=high_mid, y=n_states / 2, label="Higher\nPurchasing\nPower",
           color=GREEN, fontface="bold", size=5.5, hjust=0.5, vjust=0.5,
           alpha=1, lineheight=0.95) +
  geom_vline(xintercept=state_median, linetype="dotted", linewidth=0.8, color=ACCENT) +
  annotate("text", x=state_median + (x_hi - x_lo) * 0.012, y=n_states + 1.9, label=median_label,
           size=3, color="black", hjust=0, vjust=0) +
  geom_segment(aes(x=state_median, xend=mean_pp, yend=state_label), linewidth=0.9, alpha=0.5, color="black") +
  geom_point(aes(color=pt_color, size=pt_size)) +
  scale_color_identity() +
  scale_size_identity() +
  # coord_cartesian rather than limits=, which would drop any state below the floor
  scale_x_continuous(labels=label_dollar(scale=1e-3, suffix="k"), breaks=state_breaks) +
  coord_cartesian(xlim=c(x_lo, x_hi)) +
  scale_y_discrete(expand=expansion(add=c(0.6, 2.6))) +
  labs(title="Mean Purchasing Power by State",
       x="Mean Purchasing Power", y=NULL) +
  # no gridlines at all, so nothing crosses the two watermark labels
  theme(axis.text.y=element_text(size=7), panel.grid.major=element_blank(),
        panel.grid.minor=element_blank(),
        plot.margin=margin(10,10,10,10))

#### Collinearity of the indicator blocks

The reference group logic demonstrated in <a href="#fig-vif" class="quarto-xref">Figure 8</a> applies here as well. The additional concern was whether the year and state indicators introduced new collinearity after being added to the predictive specification. <a href="#tbl-vif-ml" class="quarto-xref">Table 3</a> revealed that they did not, as all reported variance inflation factors remained below the conventional threshold of 5. This confirmed that the indicator blocks could be retained without materially destabilizing the estimated coefficients.

In [37]:
exposure_check_cols=["routine_cognitive_share",
    "routine_manual_share",
    "non_routine_cognitive_share",
    "non_routine_manual_share"]

REFERENCE_EXPOSURE="non_routine_cognitive_share"
exposure_cols_model=[c for c in exposure_check_cols if c!=REFERENCE_EXPOSURE]
core_features=exposure_cols_model+["poverty_rate", "unemployment_rate"]

In [38]:
vif_features=core_features+year_cols_model+state_cols_model

X_vif=sm.add_constant(model_df[vif_features].astype(float))

vif_data=pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])],
})
vif_top=vif_data[vif_data["feature"].isin(year_cols_model+state_cols_model)].sort_values("VIF", ascending=False).head(10).copy()
vif_top["Indicator"]=(vif_top["feature"]
                      .str.replace("year_", "", regex=False)
                      .str.replace("state_", "State ", regex=False))
vif_tbl_df=vif_top[["Indicator","VIF"]].sort_values("VIF", ascending=False)

In [39]:
style_table(GT(vif_tbl_df.reset_index(drop=True))
  .fmt_number(columns="VIF", decimals=2)
  .cols_align(align="center", columns=["Indicator","VIF"])
  .tab_source_note("Computed on the full feature matrix after dropping the reference task group."))

In [40]:
# feature matrix, built now that the year and state indicators and the reference
# group logic above are settled; sliced using the grouped split made earlier
feature_cols_reg=core_features+year_cols_model+state_cols_model

X_ml=model_df[feature_cols_reg].astype(float)
y=model_df[target_col]

X_train, X_test=X_ml.iloc[train_idx], X_ml.iloc[test_idx]
y_train, y_test=y.iloc[train_idx], y.iloc[test_idx]

# baseline feature set (no year/state), reusing the SAME row split as the full model
feature_cols_base=exposure_cols_model+["poverty_rate", "unemployment_rate"]
X_base=model_df[feature_cols_base].astype(float)
X_train_base, X_test_base=X_base.iloc[train_idx], X_base.iloc[test_idx]

### Linear benchmark

This specification serves as a predictive benchmark rather than a second inferential model. It reuses the diagnostic insights established by the panel regression above, now applied to the training split alone. This shared split enables direct comparability with the subsequent random forest and neural network models. We present two versions of each linear model, estimated using ordinary least squares (OLS). The baseline model utilizes only the task groups with poverty and unemployment, while the full model incorporates year and state controls. Reporting both versions highlights the changes introduced by time and geography. Among the two, the full model is the one that is carried forward.

#### Baseline specification

In [41]:
X_train_base_sm=sm.add_constant(X_train_base)
ols_model_base=sm.OLS(y_train, X_train_base_sm).fit()

The baseline specification yielded an R² of 0.766. Relative to non-routine cognitive work, the coefficients for the other three task groups were negative, indicating lower purchasing power as counties shifted toward those groups. The largest negative association was observed for non-routine manual work, where a one percentage point shift was associated with approximately \$1,305 lower purchasing power. <a href="#tbl-ols-comparison" class="quarto-xref">Table 4</a> reports these estimates alongside those from the full specification.

#### Full specification

In [42]:
X_train_sm=sm.add_constant(X_train)
ols_model=sm.OLS(y_train, X_train_sm).fit()

Adding year and state indicators increased the coefficient from 0.766 to 0.871, indicating that time and geography contributed to additional variation in purchasing power. The unemployment rate coefficient shifted from approximately in the baseline specification to in the full specification for each one percentage point increase in unemployment. This change in direction suggested that the baseline estimate accounted for geographic and temporal differences that were not previously considered. In contrast, the task group coefficients maintained their direction and relative ordering.

<a href="#fig-diagnostics" class="quarto-xref">Figure 12</a> demonstrated that residual curvature persisted even after the full level specification. This figure was important because the increase in the coefficient alone suggested that incorporating year and state indicators effectively improved the model. However, the diagnostics revealed that while these variables enhanced the overall fit, they failed to eliminate the systematic pattern in the residuals. This distinction indicated that the underlying issue was the functional form of purchasing power on its original dollar scale, rather than omitted time or geographic differences.

#### Full specification on the logged outcome

<a href="#fig-binned-scale" class="quarto-xref">Figure 13</a> showed that the level specification did not fit purchasing power evenly across its range. The largest departures occurred at the lower and upper ends of the distribution, where observed purchasing power was roughly \$8,000 to \$9,000 above the fitted values. This systematic pattern suggested that the remaining problem was related to the scale of purchasing power rather than simply omitted controls, providing a clear reason to test the outcome in logarithmic form.

In [43]:
assert (y<=0).sum()==0  # confirm log is safe

y_train_log=np.log(y_train)
y_test_log=np.log(y_test)

ols_model_log=sm.OLS(y_train_log, X_train_sm).fit()

X_test_sm=sm.add_constant(X_test, has_constant="add")
y_pred_ols_log=ols_model_log.predict(X_test_sm)
y_pred_ols_log_dollars=np.exp(y_pred_ols_log)

ols_log_test_r2_log=r2_score(y_test_log, y_pred_ols_log)
ols_log_test_r2=r2_score(y_test, y_pred_ols_log_dollars)
ols_log_test_mae=mean_absolute_error(y_test, y_pred_ols_log_dollars)

Logging purchasing power reduced the residual pattern observed in <a href="#fig-diagnostics" class="quarto-xref">Figure 12</a> and enhanced the performance of the linear specification. After transforming the predictions back to dollars, the log linear model achieved an in-sample accuracy of 0.913 and a held-out R² of 0.896. This close agreement suggested that the model effectively generalized to counties not included in the training data.

#### Coefficient comparison

<a href="#tbl-specmatrix" class="quarto-xref">Table 5</a> compared the three specifications. The log specification was chosen for inference because the diagnostics for the level specification indicated that its functional form was unsuitable, making its coefficient estimates less reliable for interpretation.

In [44]:
log_coefs=ols_model_log.params[feature_cols_base]

# task groups are 0 to 1 scale (1pp=0.01 units); poverty and unemployment are already in point units
unit_per_point={
    "routine_cognitive_share": 0.01,
    "routine_manual_share": 0.01,
    "non_routine_manual_share": 0.01,
    "poverty_rate": 1.0,
    "unemployment_rate": 1.0,
}

pct_change_per_point={
    feat: (np.exp(log_coefs[feat]*unit_per_point[feat])-1)*100
    for feat in feature_cols_base
}

label_map={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_manual_share":"Non-Routine Manual",
    "poverty_rate":"Poverty Rate",
    "unemployment_rate":"Unemployment Rate",
}

task_feats={"routine_cognitive_share","routine_manual_share","non_routine_manual_share"}

comparison_ols=pd.DataFrame({
    "Group": ["Task Groups" if f in task_feats else "Economic Controls"
              for f in feature_cols_base],
    "Variable": [label_map[f] for f in feature_cols_base],
    "Baseline": ols_model_base.params[feature_cols_base].values,
    "Full": ols_model.params[feature_cols_base].values,
    "Coefficient": ols_model_log.params[feature_cols_base].values,
    "Percent per point": [pct_change_per_point[f] for f in feature_cols_base],
})
task_rows=[i for i,g in enumerate(comparison_ols["Group"]) if g=="Task Groups"]

style_table(GT(comparison_ols, rowname_col="Variable", groupname_col="Group")
  .fmt_currency(columns=["Baseline","Full"], decimals=0)
  .fmt_number(columns="Coefficient", decimals=3)
  .fmt_number(columns="Percent per point", decimals=2)
  .cols_label(**{"Percent per point": "% per 1 pp"})
  .tab_spanner(label="Purchasing Power", columns=["Baseline","Full"])
  .tab_spanner(label="Log Transformed Purchasing Power", columns=["Coefficient","Percent per point"])
  .tab_style(style=style.text(weight="bold"),
             locations=loc.body(columns="Percent per point", rows=task_rows))
  .tab_style(style=style.text(align="center", weight="bold"), locations=loc.row_groups())
  .tab_source_note("Non-routine cognitive is the reference task group. Task group coefficients in the three model columns are per unit of group proportion; the final column converts the log coefficients to the percent change in purchasing power associated with a one percentage point shift."))

<a href="#tbl-ols-comparison" class="quarto-xref">Table 4</a> revealed that the task group coefficients maintained the same direction across all three specifications, although their magnitudes changed after incorporating controls. The non-routine manual coefficient shifted from approximately (-\$130,549) in the baseline specification to (-\$91,904) in the full level specification. Routine cognitive changed from about (-\$81,085) to (-\$87,421), while routine manual changed from (-\$71,490) to (-\$81,143). These results demonstrated that adjustments altered the magnitude of the associations without altering their overall order.

The poverty rate coefficient increased in magnitude from approximately (-\$1,376) to (-\$1,729) for each additional percentage point of poverty. In contrast, the unemployment rate coefficient changed from approximately (-\$1,207) in the baseline specification to (+\$450) in the full level specification. This difference underscored that poverty remained strongly associated with purchasing power after adjustment, while the unemployment association was comparatively weak and sensitive to specification.

In [45]:
spec_summary = pd.DataFrame({
    "Metric": ["Outcome", "Year and State Controls", "R²",
               "Residual Diagnostics", "Primary Specification"],
    "Baseline": ["Purchasing Power", "No", f"{ols_model_base.rsquared:.3f}", "—", "No"],
    "Full Model": ["Purchasing Power", "Yes", f"{ols_model.rsquared:.3f}", "Failed", "No"],
    "Log Model": ["log(Purchasing Power)", "Yes", f"{ols_model_log.rsquared:.3f}", "Passed", "Yes"],
})

In [46]:
style_table(GT(spec_summary, rowname_col="Metric")
  .tab_style(style=style.text(weight="bold"),
             locations=loc.body(columns="Log Model", rows=[2, 3, 4]))
  .tab_source_note("R² is in sample, fit on the 9,565 training rows only."))

In the log specification, each additional percentage point of poverty was associated with approximately 2.9 percent lower purchasing power, assuming task groups, time, and geography remained constant. This proportional association was larger than that of any individual task group, making poverty the strongest single association among the variables reported in <a href="#tbl-ols-comparison" class="quarto-xref">Table 4</a>.

### Random forest

For the random forest model, hyperparameters were selected using a grid search with grouped five-fold cross-validation on the training counties. The search explored various configurations, including the number of trees ranging from 600 to 1,000, maximum depth from 10 to 20 or unlimited, minimum leaf size from 1 to 5, and the proportion of features considered at each split. Grouping by county ensured that the separation between places during tuning was preserved, reducing the risk of selecting parameters that performed well only due to observations from the same county appearing across folds. The final model employed 1,000 trees, unlimited depth, a minimum leaf size of 1, and half of the available features at each split.

In [47]:
# grid search actually run once; kept for documentation of the search space,
# not re-executed on render. Best params hardcoded in the following chunk.

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold

rf=RandomForestRegressor(random_state=42, n_jobs=-1)

param_grid={
    "n_estimators": [600, 1000],
    "max_depth": [None, 10, 20],
    "min_samples_leaf": [1, 2, 5],
    "max_features": [1.0, "sqrt", 0.5],
}

group_cv=GroupKFold(n_splits=5)
gcv=GridSearchCV(rf, param_grid, cv=group_cv, scoring="r2", n_jobs=-1)
gcv.fit(X_train, y_train, groups=groups_train)

In [48]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance

# grid search above was run once over the full param_grid; best params found were:
# {'max_depth': None, 'max_features': 0.5, 'min_samples_leaf': 1, 'n_estimators': 1000}
best_params={
    "max_depth": None,
    "max_features": 0.5,
    "min_samples_leaf": 1,
    "n_estimators": 1000,
}

best_rf=RandomForestRegressor(**best_params, random_state=42, n_jobs=2)
best_rf.fit(X_train, y_train)

y_pred_rf=best_rf.predict(X_test)
rf_test_r2=r2_score(y_test, y_pred_rf)
rf_test_mae=mean_absolute_error(y_test, y_pred_rf)
rf_train_r2=best_rf.score(X_train, y_train)

The random forest model achieved an R² value of 0.876 on the held-out counties, which was slightly lower compared to the log linear model’s R² of 0.896 on the same held-out counties. This 0.020 difference suggested that the additional flexibility of the random forest model did not substantially enhance its performance. Additionally, the random forest model achieved an R² value of 0.989 on the training counties, resulting in a 0.113 gap between training and held-out performance. This gap indicated that the random forest model captured additional structure in the training data that did not generalize to counties it had not encountered.

In [49]:
rf_resid_df=pd.DataFrame({"pred":y_pred_rf, "resid":y_test-y_pred_rf})

In [50]:
%%R -i rf_resid_df -w 7 -h 5 -u in -r 150 -b transparent
ggplot(rf_resid_df, aes(x=pred, y=resid)) +
  geom_point(alpha=0.3, size=0.6, color=PINK) +
  geom_hline(yintercept=0, linetype="dashed", linewidth=0.9, color="#0B0B0B") +
  scale_x_continuous(labels=dollar_axis) +
  scale_y_continuous(labels=dollar_axis) +
  labs(title="Random Forest Residuals vs. Predicted Values",
       x="Predicted Values", y="Residuals") +
  theme(panel.grid.major.y=element_blank())

<a href="#fig-rf-resid" class="quarto-xref">Figure 15</a> demonstrated no systematic curvature or trend in the random forest residuals across the range of predicted values. This suggested that the forest effectively captured the nonlinear structure that posed challenges for the level linear model without transforming purchasing power. However, the lower held-out R² value of 0.876 and the large gap between training and test performance indicated that this ability to capture the structure did not translate into improved generalization to unseen counties.

In [51]:
perm_imp=permutation_importance(
    best_rf, X_test, y_test, n_repeats=20, random_state=42, n_jobs=1
)

#### Ablation and permutation importance

In [52]:
rf_log=RandomForestRegressor(**best_params, random_state=42, n_jobs=2)
rf_log.fit(X_train, y_train_log)

y_pred_rf_log=rf_log.predict(X_test)
y_pred_rf_log_dollars=np.exp(y_pred_rf_log)

rf_log_test_r2_log=r2_score(y_test_log, y_pred_rf_log)
rf_log_test_r2=r2_score(y_test, y_pred_rf_log_dollars)
rf_log_test_mae=mean_absolute_error(y_test, y_pred_rf_log_dollars)

The random forest is largely insensitive to whether purchasing power is modeled on its original or logarithmic scale. Fitting the model to the logged outcome and then back-transforming the predictions yields an R² of 0.877, which is slightly higher than the R² of 0.876 obtained when the outcome is modeled directly. Since the log specification is also employed for the primary linear model, the log target forest is utilized for the subsequent ablations and model comparisons. The interpretability measures, including permutation and partial dependence, are computed on the raw target forest, which reports directly in dollars.

In [53]:
feature_cols_no_exposure=[c for c in feature_cols_reg if c not in exposure_cols_model]
X_train_ne=X_train[feature_cols_no_exposure]
X_test_ne=X_test[feature_cols_no_exposure]

rf_no_exposure=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_no_exposure.fit(X_train_ne, y_train_log)
rf_no_exposure_r2=r2_score(y_test, np.exp(rf_no_exposure.predict(X_test_ne)))

Ablation analysis revealed the model’s effectiveness in compensating for the removal of an entire group of variables and subsequent refitting from scratch. The removal of task groups resulted in a decline in the held out R² value from 0.876 to 0.834, a decrease of 0.042. Since the forest was retrained after removing the task groups, the remaining variables had the opportunity to recover any overlapping information. Consequently, the remaining decline indicated that poverty, unemployment, time, and geography could not fully replace the information provided by the task groups.

In [54]:
feature_cols_no_econ=[c for c in feature_cols_reg if c not in ["poverty_rate", "unemployment_rate"]]
X_train_ne2=X_train[feature_cols_no_econ]
X_test_ne2=X_test[feature_cols_no_econ]

rf_no_econ=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
rf_no_econ.fit(X_train_ne2, y_train_log)
rf_no_econ_r2=r2_score(y_test, np.exp(rf_no_econ.predict(X_test_ne2)))

In comparison, eliminating poverty and unemployment reduced the held out R² from 0.877 to 0.698, resulting in a decline of 0.179. This drop was substantially larger than the 0.042 decline observed when the task groups were removed, suggesting that the economic controls provided more of the model’s explanatory power. Even after incorporating the task groups into the economic controls, performance improved, indicating that they contributed information not fully captured by poverty and unemployment alone. All three models utilized the same forest specification, logged outcome, and held-out counties, ensuring direct comparability of their performance differences.

#### Grouped permutation importance

Joint permutation provided a complementary measure of how strongly the fitted random forest relied on the task groups. Permuting the three modeled task group proportions together reduced the held out R² by 0.153 while leaving the fitted model unchanged. This decline was much larger than the 0.042 reduction from removing the task groups and refitting the forest.

In [55]:
exposure_cols=exposure_cols_model  # the three non-reference task groups
rng=np.random.RandomState(42)

baseline_r2=r2_score(y_test, best_rf.predict(X_test))

def grouped_permutation_drop(cols, n_repeats=20):
    drops=[]
    for _ in range(n_repeats):
        X_perm=X_test.copy()
        shuffled_idx=rng.permutation(len(X_perm))
        X_perm[cols]=X_perm[cols].values[shuffled_idx]
        drops.append(baseline_r2-r2_score(y_test, best_rf.predict(X_perm)))
    return np.mean(drops)

joint_drop=grouped_permutation_drop(exposure_cols)

individual_drops={}
for col in exposure_cols:
    individual_drops[col]=grouped_permutation_drop([col])

year_drop=grouped_permutation_drop(year_cols_model)
state_drop=grouped_permutation_drop(state_cols_model)

The difference between the two measures revealed what the model could do after removing task group information. During ablation, the forest was retrained and could recover some overlapping information from poverty, unemployment, and year. Permutation, on the other hand, disrupted the task group information after the forest had already learned to use it. The larger permutation decline indicated that the fitted forest heavily relied on the task groups, even though some of their information overlapped with the other variables.

Year indicators showed a decline of 0.128 when permuted, compared to only 0.008 for state indicators. This difference demonstrated that time contributed considerably to held-out performance compared to state after accounting for economic controls and task groups.

Among the three modeled task groups, non-routine manual produced the largest individual permutation decline. These values were not interpreted as additive or independent contributions because the task groups were mechanically related through their employment proportions. Nevertheless, the ordering was consistent with the regression results, where non-routine manual also had the largest negative task group association with purchasing power.

### Neural network

The random forest model did not outperform the log linear specification, leaving us to wonder whether the lack of improvement was due to a limitation of the forest or the amount of structure present in the data. To further investigate this, we evaluated a three-hidden-layer feedforward neural network as an additional test to determine if greater model flexibility would lead to improved held-out performance. The network consisted of three hidden layers and a single linear output unit for the continuous purchasing power outcome.

In [56]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks

tf.random.set_seed(42)
np.random.seed(42)

scaler_x=StandardScaler().fit(X_train)
X_train_s=scaler_x.transform(X_train)
X_test_s=scaler_x.transform(X_test)

scaler_y=StandardScaler().fit(y_train.values.reshape(-1, 1))
y_train_s=scaler_y.transform(y_train.values.reshape(-1, 1)).ravel()
y_test_s=scaler_y.transform(y_test.values.reshape(-1, 1)).ravel()

n_features=X_train.shape[1]
h1=int(round(n_features*1.5))
h2=max(int(round(h1*0.5)), 20)
h3=10

nn=models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(h1, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(h2, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(h3, activation="relu"),
    layers.Dense(1, activation="linear"),  # regression head
])

nn.compile(optimizer="adam", loss="mse", metrics=["mae"])

es=callbacks.EarlyStopping(patience=15, restore_best_weights=True)

history=nn.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

y_pred_nn_s=nn.predict(X_test_s).ravel()
y_pred_nn=scaler_y.inverse_transform(y_pred_nn_s.reshape(-1, 1)).ravel()

nn_initial_test_r2=r2_score(y_test, y_pred_nn)
nn_initial_test_mae=mean_absolute_error(y_test, y_pred_nn)

nn_init_df=pd.DataFrame({
    "epoch": range(1, len(history.history["loss"])+1),
    "Train": history.history["loss"],
    "Validation": history.history["val_loss"],
}).melt(id_vars="epoch", var_name="series", value_name="loss")

Training loss steadily declined, while validation loss began to rise. This divergence indicated that the network was fitting the training data more closely without improving performance on unseen observations, suggesting overfitting. To address this, we added L2 weight regularization, increased dropout, and shortened the early stopping patience. These changes narrowed the gap between training and validation loss, providing a stronger test of whether the neural network could generalize beyond the training data.

In [57]:
from tensorflow.keras import regularizers

nn=models.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(h1, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(h2, activation="relu", kernel_regularizer=regularizers.l2(0.001)),
    layers.Dropout(0.3),
    layers.Dense(h3, activation="relu"),
    layers.Dense(1, activation="linear"),
])

nn.compile(optimizer="adam", loss="mse", metrics=["mae"])

es=callbacks.EarlyStopping(patience=8, restore_best_weights=True)

history=nn.fit(
    X_train_s, y_train_s,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    callbacks=[es],
    verbose=0,
)

y_pred_nn_s=nn.predict(X_test_s).ravel()
y_pred_nn=scaler_y.inverse_transform(y_pred_nn_s.reshape(-1, 1)).ravel()

nn_test_r2=r2_score(y_test, y_pred_nn)
nn_test_mae=mean_absolute_error(y_test, y_pred_nn)

nn_reg_df=pd.DataFrame({
    "epoch": range(1, len(history.history["loss"])+1),
    "Train": history.history["loss"],
    "Validation": history.history["val_loss"],
}).melt(id_vars="epoch", var_name="series", value_name="loss")

In [58]:
%%R -i nn_init_df -i nn_reg_df -w 10 -h 4.5 -u in -r 150 -b transparent
ymax <- max(c(nn_init_df$loss, nn_reg_df$loss)) * 1.05
# breaks run the whole axis rather than stopping at 0.8 and leaving the top of the
# validation curve untick-marked
y_breaks <- seq(0, ceiling(ymax*10)/10, by=0.1)

best_init <- nn_init_df[nn_init_df$series=="Validation",]
best_init_epoch <- best_init$epoch[which.min(best_init$loss)]
best_init_loss <- min(best_init$loss)
best_reg <- nn_reg_df[nn_reg_df$series=="Validation",]
best_reg_epoch <- best_reg$epoch[which.min(best_reg$loss)]
best_reg_loss <- min(best_reg$loss)

label_df1 <- nn_init_df[nn_init_df$epoch==max(nn_init_df$epoch),]
label_df2 <- nn_reg_df[nn_reg_df$epoch==max(nn_reg_df$epoch),]
label_nudge <- ymax*0.018
label_df1$label_y <- label_df1$loss + ifelse(label_df1$series=="Train", -label_nudge, label_nudge)
label_df2$label_y <- label_df2$loss + ifelse(label_df2$series=="Train", -label_nudge, label_nudge)

p1 <- ggplot(nn_init_df, aes(x=epoch, y=loss, color=series)) +
  geom_segment(aes(x=best_init_epoch, xend=best_init_epoch, y=0, yend=ymax),
               inherit.aes=FALSE, linetype="dashed", linewidth=0.35, color="black") +
  geom_line(linewidth=0.8) +
  geom_point(data=best_init[best_init$epoch==best_init_epoch,], aes(x=epoch, y=loss),
             inherit.aes=FALSE, color=ORANGE, size=2.2) +
  geom_label(data=label_df1, aes(label=series, y=label_y), hjust=-0.1, size=3.4, fontface="bold",
             fill=NA, linewidth=0, label.padding=unit(0.08, "lines"), show.legend=FALSE) +
  # the minimum sits near the left edge here, so the label reads rightward off the
  # marker instead of running off the panel and into the y axis title. Nudged
  # further right and up so the now transparent label clears the validation curve.
  annotate("label", x=best_init_epoch + 0.6, y=best_init_loss + ymax*0.26,
           label=paste0("Minimum validation loss\nepoch ", best_init_epoch),
           hjust=0, size=2.9, color="grey35", fill=NA, linewidth=0,
           label.padding=unit(0.12, "lines")) +
  scale_color_manual(values=c(Train=BLUE, Validation=ORANGE), guide="none") +
  scale_y_continuous(limits=c(0, ymax), breaks=y_breaks) +
  # limits start at 0 so the axis carries a 0 tick, even though epochs begin at 1
  scale_x_continuous(breaks=scales::pretty_breaks(n=10), limits=c(0, NA),
                     expand=expansion(mult=c(0.02, 0.28))) +
  labs(x="Epoch", y="Mean Squared Error (MSE)", title="Initial Model",
       subtitle="Patience 15")

p2 <- ggplot(nn_reg_df, aes(x=epoch, y=loss, color=series)) +
  geom_segment(aes(x=best_reg_epoch, xend=best_reg_epoch, y=0, yend=ymax),
               inherit.aes=FALSE, linetype="dashed", linewidth=0.35, color="black") +
  geom_line(linewidth=0.8) +
  geom_point(data=best_reg[best_reg$epoch==best_reg_epoch,], aes(x=epoch, y=loss),
             inherit.aes=FALSE, color=ORANGE, size=2.2) +
  geom_label(data=label_df2, aes(label=series, y=label_y), hjust=-0.1, size=3.4, fontface="bold",
             fill=NA, linewidth=0, label.padding=unit(0.08, "lines"), show.legend=FALSE) +
  # reads rightward off the dashed line and sits well above both curves, matching
  # panel A; the transparent fill means it must clear the lines rather than mask them
  annotate("label", x=best_reg_epoch + 0.8, y=best_reg_loss + ymax*0.45,
           label=paste0("Minimum validation loss\nepoch ", best_reg_epoch),
           hjust=0, size=2.9, color="grey35", fill=NA, linewidth=0,
           label.padding=unit(0.12, "lines")) +
  scale_color_manual(values=c(Train=BLUE, Validation=ORANGE), guide="none") +
  scale_y_continuous(limits=c(0, ymax), breaks=y_breaks) +
  scale_x_continuous(breaks=scales::pretty_breaks(n=10), expand=expansion(mult=c(0.02, 0.28))) +
  labs(x="Epoch", y="Mean Squared Error (MSE)", title="Regularized Model",
       subtitle="L2 Regularization, Patience 8")

p1 + p2 +
  plot_annotation(title="Training and Validation Loss, Initial vs. Regularized Network",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B"))) &
  theme(panel.grid.major.y=element_blank(), panel.grid.major.x=element_blank(),
        axis.text.y=element_text(), axis.text.x=element_text()) &
  theme(panel.grid.major=element_blank(), panel.grid.minor=element_blank())

R callback write-console: In addition:   
R callback write-console: Warning messages:
  
R callback write-console: 1:   
R callback write-console: In geom_segment(aes(x = best_init_epoch, xend = best_init_epoch,  :  
R callback write-console: 
   
R callback write-console:  All aesthetics have length 1, but the data has 40 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.
  
R callback write-console: 2:   
R callback write-console: In geom_segment(aes(x = best_reg_epoch, xend = best_reg_epoch, y = 0,  :  
R callback write-console: 
   
R callback write-console:  All aesthetics have length 1, but the data has 44 rows.
ℹ Please consider using `annotate()` or provide this layer with data containing
  a single row.
  

<a href="#fig-nn-curves" class="quarto-xref">Figure 16</a> demonstrated that regularization reduced the disparity between training and validation loss, yet the neural network still failed to surpass the random forest or the log linear model. On the same counties, the random forest achieved a held-out R² of 0.877, while the log linear model achieved 0.896. Consequently, the neural network’s additional flexibility did not offer any improvement over the simpler alternatives. Since the study aimed to elucidate the relationship between task groups and purchasing power, further increases in model complexity were deemed unnecessary. Therefore, the log linear and random forest models were retained for the subsequent comparisons.

### Cross validation

Five-fold grouped cross-validation was employed to assess the consistency of the observed performance in the single train and test split across various county sets. Counties remained grouped within each fold, ensuring that observations from the same county were not included in both training and validation data.

In [59]:
from sklearn.model_selection import GroupKFold

group_kfold_cv=GroupKFold(n_splits=5)
cv_results={
    "Ordinary Least Squares": [],
    "Random Forest": [],
}

y_log=np.log(y)  # full data log target, same idea as y_train_log/y_test_log

for fold, (tr_idx, val_idx) in enumerate(group_kfold_cv.split(X_ml, y, groups)):
    X_tr, X_val=X_ml.iloc[tr_idx], X_ml.iloc[val_idx]
    y_tr, y_val=y.iloc[tr_idx], y.iloc[val_idx]
    y_tr_log=y_log.iloc[tr_idx]

    X_tr_sm=sm.add_constant(X_tr)
    X_val_sm=sm.add_constant(X_val, has_constant="add")
    ols_fold_log=sm.OLS(y_tr_log, X_tr_sm).fit()
    pred_ols=np.exp(ols_fold_log.predict(X_val_sm))  # back transformed, dollar scale
    cv_results["Ordinary Least Squares"].append(r2_score(y_val, pred_ols))

    rf_fold=RandomForestRegressor(**best_params, random_state=42, n_jobs=-1)
    rf_fold.fit(X_tr, y_tr_log)
    pred_rf=np.exp(rf_fold.predict(X_val))
    cv_results["Random Forest"].append(r2_score(y_val, pred_rf))

cv_folds=pd.DataFrame([
    {"model": m, "fold": i+1, "r2": s}
    for m, scores in cv_results.items()
    for i, s in enumerate(scores)
])

cv_summary=pd.DataFrame({m: {"Mean R²": np.mean(s), "SD": np.std(s)}
                         for m, s in cv_results.items()}).T.reset_index()
cv_summary.columns=["Model","Mean R²","SD"]

In [60]:
style_table(GT(cv_summary)
  .fmt_number(columns=["Mean R²","SD"], decimals=4)
  .cols_align(align="right", columns=["Mean R²","SD"])
  .cols_width(cases={"Model": "130px", "Mean R²": "90px", "SD": "90px"})
  .tab_source_note("Both models are fit on the log target and scored on the dollar scale after back transformation."))

/opt/anaconda3/envs/capstone/lib/python3.12/site-packages/great_tables/_render_checks.py:37: RenderWarning: Rendering table with .cols_width() in Quarto may result in unexpected behavior. This is because Quarto performs custom table processing. Either use all percentage widths, or set .tab_options(quarto_disable_processing=True) to disable Quarto table processing.
  warnings.warn(

The <a href="#tbl-cv-summary" class="quarto-xref">Table 6</a> revealed that the log linear model and random forest achieved comparable mean values across the five folds. The difference in their average performance was smaller than the variation observed across folds, suggesting that the earlier comparison was not influenced by a specific train and test split. Consequently, the neural network was excluded as it failed to enhance held-out performance and was not retained.

## Summary

The panel regression provided the primary inferential results, with standard errors clustered by county and controlling for poverty, unemployment, population, and year. Diagnostic checks supported a proportional relationship, justifying the log specification. However, the variance decomposition revealed that most task group variation occurred between counties rather than within counties over time. Predictive models were evaluated on held-out counties, where the log linear model achieved an R² of 0.896 compared to 0.877 for the random forest. The neural network model provided no further improvement. Removing the task groups reduced the held-out R² by 0.042, indicating that poverty, unemployment, time, and geography did not fully capture the information they provided. These results collectively support retaining the log linear specification as the primary representation of the relationship reported in <a href="#sec-results" class="quarto-xref">Section 5</a>.

# Results

In [61]:
# inbound from _04: reg, df, model, model_log, X_ml, X_test, y_test, y_pred_rf,
# y_pred_rf_log_dollars, best_rf, perm_imp, joint_drop, year_drop, state_drop,
# cv_folds, engine, ols_log_test_r2, rf_log_test_r2, nn_test_r2,
# ols_log_test_mae, rf_log_test_mae, nn_test_mae
import pandas as pd
import numpy as np
import statsmodels.api as sm
from great_tables import GT, html

res_cols=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
          "poverty_rate","unemployment_rate","log_population"]

mean_pp=float(reg["affordability_salary"].mean())
group_terms=["non_routine_manual_share","routine_cognitive_share","routine_manual_share"]
group_names=["Non-Routine Manual","Routine Cognitive","Routine Manual"]

The results unveiled four main findings. Task groups maintained their association with purchasing power even after accounting for factors like poverty, unemployment, population, and year. A one percentage point shift from non-routine cognitive to non-routine manual work was linked to a reduction in purchasing power of approximately \$846, with a 95 percent confidence interval of ±\$38 at the panel mean. The log specification provided the most comprehensive representation of this relationship, achieving a held out R² of 0.896, compared to 0.877 for the random forest, while the neural network offered no further improvement. Poverty rate demonstrated the strongest association with purchasing power, but task groups provided additional insights beyond conventional economic indicators. Moreover, differences in task groups persisted across counties and over time. The subsequent sections delve into the magnitude, geographical distribution, and stability of these relationships.

## The association

In [62]:
# log specification coefficients (model_log fit in _04, cluster robust), expressed as
# dollars at the panel mean per one percentage point shift
coef_rows=[]
for term, name in zip(group_terms, group_names):
    b=model_log.params[term]
    se=model_log.bse[term]
    est=(np.exp(b*0.01)-1)*mean_pp
    lo=(np.exp((b-1.96*se)*0.01)-1)*mean_pp
    hi=(np.exp((b+1.96*se)*0.01)-1)*mean_pp
    coef_rows.append({"group":name, "est":est, "lo":lo, "hi":hi})
coef_df=pd.DataFrame(coef_rows)

The log specification of the panel regression was employed for inference based on the diagnostic results outlined in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. A one percentage point shift from non-routine cognitive to non-routine manual work was associated with approximately \$846 lower purchasing power, with a 95 percent confidence interval of ±\$38 at the panel mean (<a href="#fig-coefplot" class="quarto-xref">Figure 17</a>). Routine cognitive and routine manual work also exhibited negative associations with purchasing power, although their estimated differences were smaller. This ordering aligns with the level specification reported in <a href="#sec-analysis" class="quarto-xref">Section 4</a>. For comparison, each additional percentage point of poverty was linked to approximately 2.9 percent lower purchasing power, a larger proportional association than any individual task group.

In [63]:
%%R -i coef_df -w 8 -h 3.5 -u in -r 150 -b transparent
dollar_axis <- function(v) ifelse(is.na(v), "", ifelse(v == 0, "$0",
  sprintf("%s$%s", ifelse(v < 0, "-", ""), formatC(abs(v), format="d", big.mark=","))))
coef_df$group <- factor(coef_df$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))

ggplot(coef_df, aes(x=est, y=group, color=group)) +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.9, color="#0B0B0B") +
  geom_errorbar(aes(xmin=lo, xmax=hi), orientation="y", width=0.15) +
  geom_point(size=2.6) +
  geom_text(aes(label=dollar_axis(est)), vjust=-1.4, size=3.3, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  # denser ticks on the same dollar scale; horizontal gridlines dropped because each
  # row is read across to the x scale, not compared vertically
  scale_x_continuous(labels=dollar_axis, breaks=scales::pretty_breaks(n=10),
                     expand=expansion(mult=c(0.1, 0.15))) +
  labs(title="Task Group Coefficients, Log Specification",
       x="Purchasing Power Change ($ per 1 pp)", y=NULL) +
  theme(panel.grid.major.y=element_blank())

<a href="#fig-coefplot" class="quarto-xref">Figure 17</a> illustrated the estimated purchasing power difference associated with a one percentage point shift from non-routine cognitive work toward each of the other task groups. All three estimates remained below zero, with non-routine manual work showing the largest negative association. The narrow intervals around the estimates indicated that the direction and ordering of these task group relationships were precisely estimated after accounting for the other variables in the model.

The level specification provided an additional perspective on the task group associations and economic controls in dollar terms. Among the controls, the poverty rate carried the largest and most precisely estimated coefficient, while the unemployment rate and log population were smaller but remained distinguishable from zero at the 0.05 level (<a href="#tbl-panel-coefs" class="quarto-xref">Table 7</a>). The table also reported the model’s overall F test, sample size, and , offering a summary of the fit and precision of the complete level specification.

In [64]:
level_coef_terms=["routine_cognitive_share","routine_manual_share","non_routine_manual_share",
                   "poverty_rate","unemployment_rate","log_population"]
level_coef_labels={
    "routine_cognitive_share":"Routine Cognitive",
    "routine_manual_share":"Routine Manual",
    "non_routine_manual_share":"Non-Routine Manual",
    "poverty_rate":"Poverty Rate",
    "unemployment_rate":"Unemployment Rate",
    "log_population":"Log Population",
}
level_coef_df=pd.DataFrame({
    "Variable": [level_coef_labels[t] for t in level_coef_terms],
    "Coefficient": [model.params[t] for t in level_coef_terms],
    "Std. Error": [model.bse[t] for t in level_coef_terms],
    "t": [model.tvalues[t] for t in level_coef_terms],
    "p-value": [model.pvalues[t] for t in level_coef_terms],
})
level_f_note=(f"F({int(model.df_model)}, {int(model.df_resid)}) = {model.fvalue:.1f}, "
              f"p {'< 0.001' if model.f_pvalue < 0.001 else f'= {model.f_pvalue:.3f}'}; "
              f"n = {int(model.nobs):,}; R² = {model.rsquared:.3f}. "
              "Standard errors clustered by county.")

In [65]:
style_table(GT(level_coef_df)
  .fmt_currency(columns="Coefficient", decimals=0)
  .fmt_number(columns="Std. Error", decimals=0, use_seps=True)
  .fmt_number(columns="t", decimals=2)
  .fmt_number(columns="p-value", decimals=3)
  .tab_source_note(level_f_note))

## Not poverty in disguise

A natural concern is that the task group associations simply reflect poverty, as counties with higher poverty rates also tend to have more routine work. <a href="#fig-baseline-maps" class="quarto-xref">Figure 18</a> showed the geographic distribution of mean poverty and unemployment rates across the study period. Both measures were highest across much of the rural South, the Mississippi Delta, and parts of the Southwest border region, which overlapped with areas of low purchasing power in <a href="#fig-afford-map" class="quarto-xref">Figure 23</a>. This overlap was important because each additional percentage point of poverty was associated with approximately 2.9 percent lower purchasing power in the log specification. Unemployment also showed a concentration along the California coast and Central Valley that was less apparent for poverty, indicating that the two measures captured different dimensions of local economic conditions.

In [66]:
import geopandas as gpd

map_baseline=pd.read_sql("""
    select county_fips, avg(poverty_rate) as poverty_rate, avg(unemployment_rate) as unemployment_rate
    from county_baseline
    group by county_fips
""", engine)
map_baseline['fips']=map_baseline['county_fips'].astype(str).str.zfill(5)

gdf_baseline=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_baseline=gdf_baseline.rename(columns={'GEOID':'fips'})
merged_baseline=gdf_baseline.merge(map_baseline[['fips','poverty_rate','unemployment_rate']], on='fips', how='left')
merged_baseline=merged_baseline[~merged_baseline['STATEFP'].isin(['02','15','60','66','69','72','78'])]
os.makedirs("output", exist_ok=True)
merged_baseline[['fips','poverty_rate','unemployment_rate','geometry']].to_file("output/baseline_maps.geojson", driver="GeoJSON")

pov_vmin=float(merged_baseline['poverty_rate'].quantile(0.02))
pov_vmax=float(merged_baseline['poverty_rate'].quantile(0.98))
unemp_vmin=float(merged_baseline['unemployment_rate'].quantile(0.02))
unemp_vmax=float(merged_baseline['unemployment_rate'].quantile(0.98))

In [67]:
%%R -i pov_vmin -i pov_vmax -i unemp_vmin -i unemp_vmax -w 11 -h 5.5 -u in -r 150 -b transparent
suppressMessages(library(sf))

baseline_sf <- st_read("output/baseline_maps.geojson", quiet=TRUE)

p1 <- ggplot(baseline_sf) +
  geom_sf(aes(fill=poverty_rate), color="white", linewidth=0.05) +
  scale_fill_distiller(palette="YlOrBr", direction=1, limits=c(pov_vmin, pov_vmax), oob=scales::squish,
                        na.value="#eeeeee", name="Poverty Rate (%)") +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.2, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA)) +
  labs(title="Poverty Rate")

p2 <- ggplot(baseline_sf) +
  geom_sf(aes(fill=unemployment_rate), color="white", linewidth=0.05) +
  scale_fill_distiller(palette="YlOrBr", direction=1, limits=c(unemp_vmin, unemp_vmax), oob=scales::squish,
                        na.value="#eeeeee", name="Unemployment Rate (%)") +
  coord_sf(crs=st_crs(5070)) +
  theme_void() +
  theme(legend.position="bottom", legend.key.width=unit(1.2, "cm"),
        plot.title=element_text(face="bold", size=11, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA)) +
  labs(title="Unemployment Rate")

p1 + p2 +
  plot_annotation(title="Two Conventional Distress Measures, by County",
                  tag_levels="A", tag_suffix=")",
                  theme=theme(plot.title=element_text(face="bold", size=13, hjust=0.5),
                              plot.tag=element_text(face="bold", size=11, color="#0B0B0B")))

<a href="#fig-coefcompare" class="quarto-xref">Figure 19</a> conducted a test to determine if the economic conditions accounted for the associations between task groups. They compared two log panel specifications estimated on the same observations. The first specification included task groups and year indicators, while the second added poverty, unemployment, and log population. After adding the controls, the estimates for the non-routine manual and routine manual became smaller, indicating that some of their initial associations overlapped with these economic conditions. However, routine cognitive remained relatively stable, staying close to −1.1 percent in both specifications. Notably, all three task group estimates remained negative and accurately estimated after adjustment. This comparison demonstrated that poverty, unemployment, and population explained part of the relationship, but they did not fully account for the association between task groups and purchasing power.

In [68]:
share_terms=["routine_cognitive_share","routine_manual_share","non_routine_manual_share"]

X_nocontrols=pd.concat([reg[share_terms],
                        pd.get_dummies(reg["year"], prefix="year", drop_first=True).astype(float)], axis=1)
X_nocontrols=sm.add_constant(X_nocontrols)
model_nc_log=sm.OLS(reg["actual_log"], X_nocontrols).fit(
    cov_type="cluster", cov_kwds={"groups": reg["county_fips"]})

def pct_ci(m, term):
    b=m.params[term]
    se=m.bse[term]
    est=(np.exp(b*0.01)-1)*100
    lo=(np.exp((b-1.96*se)*0.01)-1)*100
    hi=(np.exp((b+1.96*se)*0.01)-1)*100
    return est, lo, hi

compare_rows=[]
for term, name in zip(share_terms, ["Routine Cognitive","Routine Manual","Non-Routine Manual"]):
    for m, spec in [(model_nc_log, "Baseline"), (model_log, "With controls")]:
        est, lo, hi = pct_ci(m, term)
        compare_rows.append({"group":name, "spec":spec, "est":est, "lo":lo, "hi":hi})
compare_df=pd.DataFrame(compare_rows)

compare_wide=compare_df.pivot(index="group", columns="spec", values="est").reset_index()
compare_wide.columns=["group","est_baseline","est_controls"]

In [69]:
%%R -i compare_df -w 9 -h 4.5 -u in -r 150 -b transparent
compare_df$group <- factor(compare_df$group,
    levels=rev(c("Non-Routine Manual","Routine Cognitive","Routine Manual")))
compare_df$spec <- factor(compare_df$spec,
    levels=c("Baseline","With controls"))

# the two specifications sit on their own y offsets within each task group, so the
# estimates and their intervals never overlap even where the two barely differ
dodge <- position_dodge(width=0.55)
lab_pad <- diff(range(c(compare_df$lo, compare_df$hi))) * 0.02

ggplot(compare_df, aes(x=est, xmin=lo, xmax=hi, y=group, color=spec, shape=spec)) +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.9, color="#0B0B0B") +
  geom_errorbar(orientation="y", position=dodge, width=0.18, linewidth=0.6) +
  geom_point(position=dodge, fill="white", size=3.2, stroke=1.0) +
  geom_text(aes(x=hi + lab_pad, label=sprintf("%.1f%%", est)), position=dodge,
            hjust=0, size=3.2, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=c("Baseline"=ORANGE, "With controls"=GREEN)) +
  scale_shape_manual(values=c("Baseline"=21, "With controls"=19)) +
  # the percent sign moves onto the tick values, so the axis title stays clean
  scale_x_continuous(breaks=scales::pretty_breaks(n=10),
                     labels=function(v) sprintf("%g%%", v),
                     expand=expansion(mult=c(0.04, 0.10))) +
  scale_y_discrete(expand=expansion(add=0.6)) +
  labs(title="Task Group Coefficients Before and After Controls",
       x="Purchasing Power Change", y=NULL, color=NULL, shape=NULL) +
  theme(legend.position="bottom", panel.grid.major.y=element_blank())

## What carries the signal

Two complementary tests were conducted to assess the contribution of information from the task groups. The <a href="#fig-ablation" class="quarto-xref">Figure 20</a> test demonstrated the impact of removing an entire feature block and refitting the random forest. Removing poverty and unemployment resulted in a 0.178 reduction in held out R², compared to a smaller decline of 0.042 when the task groups were removed. This indicates that the economic controls provided more explanatory information overall, but the task groups still improved performance beyond what poverty and unemployment alone could capture.

In [70]:
ablation_df=pd.DataFrame({
    "model": ["Task Groups","Economic Controls"],
    "r2": [0.834, 0.698],
})
ablation_df["full_r2"]=0.876
ablation_df["drop"]=ablation_df["full_r2"]-ablation_df["r2"]

In [71]:
%%R -i ablation_df -w 7 -h 3 -u in -r 150 -b transparent
ablation_df$model <- factor(ablation_df$model, levels=c("Task Groups","Economic Controls"))

ggplot(ablation_df, aes(x=drop, y=model, fill=model)) +
  geom_col(width=0.55) +
  geom_text(aes(label=sprintf("%.3f", drop)), hjust=-0.25, size=4.0, fontface="bold", color="#252525") +
  scale_fill_manual(values=c("Task Groups"=GREEN, "Economic Controls"=ORANGE), guide="none") +
  scale_x_continuous(limits=c(0, 0.20), breaks=seq(0, 0.20, 0.05),
                     labels=function(v) sprintf("%.2f", v), expand=c(0,0)) +
  labs(title="Held Out R² Loss From Feature Block Removal",
       x="R² Loss", y=NULL) +
  theme(panel.grid.major.y=element_blank())

The <a href="#fig-importance" class="quarto-xref">Figure 21</a> test offered a second measure by leaving the fitted model unchanged and perturbing each feature or feature block through permutation. Jointly permuting the task groups led to a 0.153 reduction in held out R², compared to 0.128 for the year indicators and 0.008 for the state indicators. Poverty exhibited the largest individual decline. These results suggest that the fitted forest heavily relied on the task groups even after incorporating economic controls, time, and geography.

The ablation and permutation results provided further insights into the model’s performance. Removing the task groups and refitting the model resulted in a modest reduction of 0.042 in the held-out R² value, suggesting that the model could still recover some overlapping information from poverty, unemployment, and year. In contrast, permuting the task groups after fitting caused a decline of 0.153, indicating that the model could no longer compensate for the loss of information. These two tests together revealed that the task groups contained information that overlapped with other variables but was not fully replaced by them.

In [72]:
pov_idx=list(X_test.columns).index("poverty_rate")
unemp_idx=list(X_test.columns).index("unemployment_rate")

importance_df=pd.DataFrame({
    "feature": ["Poverty Rate","Task Groups","Year",
                "Unemployment Rate","State"],
    "block": ["Economic Controls","Task Groups","Structural Indicators",
              "Economic Controls","Structural Indicators"],
    "drop": [perm_imp.importances_mean[pov_idx], joint_drop, year_drop,
             perm_imp.importances_mean[unemp_idx], state_drop],
}).sort_values("drop")

In [73]:
%%R -i importance_df -w 8 -h 3.5 -u in -r 150 -b transparent
importance_df$feature <- factor(importance_df$feature, levels=importance_df$feature)
importance_df$block <- factor(importance_df$block,
    levels=c("Task Groups","Economic Controls","Structural Indicators"))

ggplot(importance_df, aes(x=drop, y=feature, fill=block)) +
  geom_col(width=0.6) +
  geom_text(aes(label=sprintf("%.3f", drop)), hjust=-0.25, size=3.4, fontface="bold", color="#252525") +
  scale_fill_manual(values=c("Task Groups"=GREEN, "Economic Controls"=ORANGE,
                             "Structural Indicators"=GREY)) +
  # ticks every 0.05; ggplot keeps only those inside the existing range, so the
  # scale itself is unchanged
  scale_x_continuous(breaks=seq(0, 1, 0.05), labels=function(v) sprintf("%.2f", v),
                     expand=expansion(mult=c(0, 0.15))) +
  labs(title="Feature Importance by Grouped Permutation",
       x="Drop in Held Out R² When Permuted", y=NULL, fill=NULL) +
  theme(legend.position="bottom", panel.grid.major.y=element_blank())

<a href="#fig-pdp" class="quarto-xref">Figure 22</a> demonstrated how the random forest depicted the relationship across the observed range of each non-routine cognitive reference comparison. Purchasing power gradually declined across the three modeled task groups, without any distinct thresholds or reversals. Despite capturing these flexible relationships, the random forest achieved a held out R² of 0.877, compared to 0.896 for the log linear model, a difference of 0.019. This pattern aligned with the proportional relationship identified by the log linear specification and explained why the additional flexibility of the random forest did not enhance held out performance. Since the four task groups collectively accounted for the entire range, these curves represented the model’s response to changes in one group while keeping the remaining values constant. Therefore, these curves should not be interpreted as independent changes in employment.

In [74]:
from sklearn.inspection import partial_dependence

pdp_frames=[]
for term, name in zip(["routine_cognitive_share","routine_manual_share","non_routine_manual_share"],
                      ["Routine Cognitive","Routine Manual","Non-Routine Manual"]):
    pd_res=partial_dependence(best_rf, X_test, [term], kind="average", grid_resolution=40)
    grid=pd_res["grid_values"][0] if "grid_values" in pd_res else pd_res["values"][0]
    pdp_frames.append(pd.DataFrame({"x":grid,
                                    "y":pd_res["average"][0],
                                    "group":name}))
pdp_df=pd.concat(pdp_frames, ignore_index=True)

In [75]:
%%R -i pdp_df -w 9.5 -h 3.8 -u in -r 150 -b transparent
pdp_levels <- c("Routine Cognitive","Routine Manual","Non-Routine Manual")
pdp_df$group <- factor(pdp_df$group, levels=pdp_levels)
PDP_COLORS <- TASK_COLORS[pdp_levels]

# the strip carries the group name alone, centred; the letter tag is drawn separately
# above the top of each panel's own y axis, so the two never share one string
tag_df <- data.frame(group=factor(pdp_levels, levels=pdp_levels),
                     tag=paste0(LETTERS[seq_along(pdp_levels)], ")"))

ggplot(pdp_df, aes(x=x, y=y, color=group)) +
  geom_line(linewidth=1.0) +
  geom_text(data=tag_df, aes(x=-Inf, y=Inf, label=tag), inherit.aes=FALSE,
            hjust=1.2, vjust=-0.45, fontface="bold", size=4.2, color="#0B0B0B") +
  # axes="all_y" repeats the y axis on every panel; the scale itself stays shared,
  # so the three declines remain comparable in magnitude
  facet_wrap(~group, ncol=3, axes="all_y") +
  coord_cartesian(clip="off") +
  scale_color_manual(values=PDP_COLORS, guide="none") +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(round(v), format="d", big.mark=",")),
                     breaks=scales::pretty_breaks(n=7),
                     expand=expansion(mult=c(0.02, 0.02))) +
  scale_x_continuous(labels=function(v) v * 100, breaks=scales::pretty_breaks(n=6)) +
  labs(title="Partial Dependence on Each Task Group",
       x="Shift from Non-Routine Cognitive (Percentage Points)",
       y="Predicted Purchasing Power") +
  theme(panel.spacing=unit(1.2, "lines"), plot.margin=margin(16, 10, 8, 10))

## Where purchasing power is strained

The association also exhibited a distinct geographic pattern. <a href="#fig-afford-map" class="quarto-xref">Figure 23</a> revealed that purchasing power was highest along the metropolitan Northeast corridor, across parts of the upper Midwest, and in certain regions of the mountain West. Conversely, the lowest values were concentrated across the rural South and the southern border region. The map included all counties with a purchasing power value, rather than only those in the analytical panel, because the outcome required only household income and Regional Price Parity.

In [76]:
import geopandas as gpd

map_all=pd.read_sql("""
    select county_fips, avg(affordability_salary) as purchasing_power
    from county_affordability
    group by county_fips
""", engine)
map_all['fips']=map_all['county_fips'].astype(str).str.zfill(5)

gdf_map=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_map=gdf_map.rename(columns={'GEOID':'fips'})

afford_vmin=float(map_all['purchasing_power'].quantile(0.02))
afford_vmax=float(map_all['purchasing_power'].quantile(0.98))

merged_map=gdf_map.merge(map_all[['fips','purchasing_power']], on='fips', how='left')
merged_map=merged_map[~merged_map['STATEFP'].isin(['02','15','60','66','69','72','78'])]
os.makedirs("output", exist_ok=True)
merged_map[['fips','purchasing_power','geometry']].to_file("output/afford_map.geojson", driver="GeoJSON")

# dominant task group per county: whichever of the four groups holds the largest mean share
dominant_share_cols=["routine_cognitive_share","routine_manual_share",
                     "non_routine_cognitive_share","non_routine_manual_share"]
dominant_label_map={"routine_cognitive_share":"Routine Cognitive","routine_manual_share":"Routine Manual",
                    "non_routine_cognitive_share":"Non-Routine Cognitive","non_routine_manual_share":"Non-Routine Manual"}
dominant_df=pd.read_sql(f"""
    select cte.county_fips,
           avg(cte.routine_cognitive_share) as routine_cognitive_share,
           avg(cte.routine_manual_share) as routine_manual_share,
           avg(cte.non_routine_cognitive_share) as non_routine_cognitive_share,
           avg(cte.non_routine_manual_share) as non_routine_manual_share,
           avg(ca.affordability_salary) as purchasing_power
    from county_task_exposure cte
    join county_affordability ca on cte.county_fips=ca.county_fips and cte.year=ca.year
    group by cte.county_fips
""", engine)
dominant_df["dominant_group"]=dominant_df[dominant_share_cols].idxmax(axis=1).map(dominant_label_map)

dominant_n=dominant_df["dominant_group"].value_counts()
dominant_med=dominant_df.groupby("dominant_group")["purchasing_power"].median()

This geographic pattern was also reflected in county task groups. Counties were categorized based on the task group that accounted for the largest average proportion of employment over the study period. In 664 counties, non-routine cognitive work was the dominant type of work, with a median purchasing power of \$61,162. In contrast, 97 non-routine manual counties had a median purchasing power of \$46,657, while 87 routine manual counties had a median purchasing power of \$50,368. Routine cognitive work was the dominant group in 0 counties. These differences demonstrated that the association observed across the continuous task group measures was also evident when counties were grouped by the type of work that constituted the largest portion of local employment.

In [77]:
%%R -i afford_vmin -i afford_vmax -w 9 -h 6 -u in -r 150 -b transparent
suppressMessages(library(sf))

afford_sf <- st_read("output/afford_map.geojson", quiet=TRUE)

DIVERGING_COLORS <- c("#d53e4f", "#fc8d59", "#fee08b", "#e6f598", "#99d594", "#3288bd")

ggplot(afford_sf) +
  geom_sf(aes(fill=purchasing_power), color="#FAFAF8", linewidth=0.05) +
  scale_fill_gradientn(colors=DIVERGING_COLORS, limits=c(afford_vmin, afford_vmax),
                        oob=scales::squish, na.value="#D9D9D6",
                        labels=label_dollar(scale=1e-3, suffix="k"),
                        name="Mean Purchasing Power") +
  coord_sf(crs=st_crs(5070), datum=NA) +
  guides(fill=guide_colorbar(direction="horizontal", title.position="top", title.hjust=0.5,
                             barwidth=unit(4.5,"cm"), barheight=unit(0.35,"cm"))) +
  labs(title="Where Purchasing Power Is Strained") +
  theme_void() +
  theme(legend.position="bottom",
        legend.title=element_text(size=10, face="bold", color="#333333"),
        legend.text=element_text(size=9, color="#555555"),
        plot.title=element_text(face="bold", size=13, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA))

## The relationship is proportional

Across the five grouped cross-validation folds, the log linear model and random forest performed similarly in terms of the variation across folds (<a href="#fig-modelcomp" class="quarto-xref">Figure 24</a>). The single held-out split also demonstrated the same pattern. The log linear model achieved an R² of 0.896, while the random forest achieved an R² of 0.877. <a href="#fig-nn-curves" class="quarto-xref">Figure 16</a> extended this comparison to the neural network, which achieved a held-out R² of 0.869 and a mean absolute error of \$4,145. The neural network did not provide any improvement over the simpler models. Overall, these results indicated that additional model flexibility did not capture enough new structure to improve performance beyond the log linear specification.

In [78]:
cv_means=cv_folds.groupby("model", as_index=False)["r2"].mean().rename(columns={"r2":"mean_r2"})

In [79]:
%%R -i cv_folds -i cv_means -w 8 -h 4 -u in -r 150 -b transparent
cv_means$mean_label <- sprintf("%.3f", cv_means$mean_r2)
cv_means$label_x <- ifelse(cv_means$model=="Ordinary Least Squares", 0.74, 2.26)

# evenly spread each fold's point off the shared category center, so the five folds
# per model don't stack into one vertical clump; the y values (and axis scale) are
# untouched, only the x position within each category changes
cv_folds$model_num <- as.numeric(factor(cv_folds$model, levels=c("Ordinary Least Squares","Random Forest")))
fold_ids <- sort(unique(cv_folds$fold))
fold_offset <- setNames(seq(-0.15, 0.15, length.out=length(fold_ids)), fold_ids)
cv_folds$x_spread <- cv_folds$model_num + fold_offset[as.character(cv_folds$fold)]
cv_means$model_num <- as.numeric(factor(cv_means$model, levels=c("Ordinary Least Squares","Random Forest")))

ggplot(cv_folds, aes(x=x_spread, y=r2, color=model)) +
  geom_line(aes(x=x_spread, y=r2, group=fold), inherit.aes=FALSE, color="grey75", linewidth=0.5) +
  geom_point(size=2.6) +
  geom_crossbar(data=cv_means, aes(x=model_num, y=mean_r2, ymin=mean_r2, ymax=mean_r2),
                width=0.35, linewidth=0.5, color="grey30", inherit.aes=FALSE) +
  geom_text(data=cv_means, aes(x=label_x, y=mean_r2, label=mean_label, color=model),
            size=3.4, fontface="bold", vjust=0.5, show.legend=FALSE) +
  scale_color_manual(values=c("Ordinary Least Squares"=BLUE,
                              "Random Forest"=PINK)) +
  scale_x_continuous(breaks=c(1,2), labels=c("Ordinary Least Squares","Random Forest"),
                      expand=expansion(add=0.6)) +
  # zoomed to the band the folds occupy. coord_cartesian rather than limits= so no
  # fold is dropped; note this magnifies a mean gap of roughly 0.006, which the
  # caption and surrounding prose both state in numbers rather than leaving to the eye
  scale_y_continuous(breaks=seq(0.875, 0.975, 0.025)) +
  coord_cartesian(ylim=c(0.875, 0.975)) +
  labs(title="Held Out R² by Fold, Linear Model vs. Random Forest",
       x=NULL, y="Held Out R²") +
  guides(color="none")

In [80]:
metrics_df=pd.DataFrame({
    "Model": ["Ordinary Least Squares","Random Forest","Neural Network"],
    "R²": [ols_log_test_r2, rf_log_test_r2, nn_test_r2],
    "Mean Absolute Error": [ols_log_test_mae, rf_log_test_mae, nn_test_mae],
})

style_table(GT(metrics_df)
  .fmt_number(columns="R²", decimals=3)
  .fmt_currency(columns="Mean Absolute Error", decimals=0)
  .tab_source_note("Ordinary Least Squares and Random Forest are fit on the log target and scored on the dollar scale after back transformation.")
  .tab_source_note("Neural network results are from this single split only; the five fold grouped cross validation table in the analysis section reports cross validated results for the other two models."))

<a href="#fig-binned" class="quarto-xref">Figure 25</a> presented a complementary perspective using purchasing power dollars. Across ten equal-sized bins of routine cognitive tasks, observed purchasing power increased in the lower range, peaked near 20 percent, and then declined. The random forest closely reproduced this pattern, rather than identifying a substantially different relationship. This observation did not contradict the smoother declines in <a href="#fig-pdp" class="quarto-xref">Figure 22</a>, as the partial dependence curves varied one task group while holding the others constant, whereas the binned comparison reflected how the task groups occurred together in actual counties. Since the four groups summed to one, changes in routine cognitive work in the observed data were accompanied by changes in the other groups. Therefore, the agreement between the flexible model and the simpler log linear specification supported retaining the proportional specification rather than adding further model complexity.

In [81]:
binned_ml=pd.DataFrame({
    "rc": X_test["routine_cognitive_share"].values,
    "Actual": y_test.values,
    "Random Forest": y_pred_rf_log_dollars,
})
binned_ml["bin"]=pd.qcut(binned_ml["rc"], 10, labels=False)
binned_means=binned_ml.groupby("bin")[["rc","Actual","Random Forest"]].mean().reset_index(drop=True)
binned_means=binned_means.rename(columns={"Actual":"Observed","Random Forest":"Predicted"})
binned_long=binned_means.melt(id_vars="rc", value_vars=["Observed","Predicted"],
                              var_name="series", value_name="pp")

In [82]:
%%R -i binned_long -i rf_log_test_r2 -i rf_log_test_mae -w 9 -h 4.5 -u in -r 150 -b transparent
OBSERVED <- "#252525"
binned_long$rc_pct <- binned_long$rc*100
label_pts <- binned_long[binned_long$rc==max(binned_long$rc),]
rf_stat_label <- sprintf("R² = %.3f\nMAE = $%s",
                          rf_log_test_r2, formatC(round(rf_log_test_mae), format="d", big.mark=","))

ggplot(binned_long, aes(x=rc_pct, y=pp, color=series)) +
  annotate("text", x=max(binned_long$rc_pct), y=max(binned_long$pp)*0.975,
           label=rf_stat_label, hjust=1, size=2.9, color="#52514E") +
  geom_line(aes(linewidth=series)) +
  geom_point(aes(size=series)) +
  geom_text(data=label_pts, aes(label=series), hjust=-0.15, size=3.4, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=c("Observed"=OBSERVED, "Predicted"=BLUE), guide="none") +
  scale_linewidth_manual(values=c("Observed"=1.0, "Predicted"=0.8), guide="none") +
  scale_size_manual(values=c("Observed"=1.8, "Predicted"=1.5), guide="none") +
  scale_x_continuous(labels=function(v) sprintf("%d%%", round(v)),
                      breaks=scales::pretty_breaks(n=8),
                      expand=expansion(mult=c(0.02, 0.28))) +
  scale_y_continuous(labels=function(v) sprintf("$%s", formatC(round(v), format="d", big.mark=",")),
                     expand=expansion(mult=c(0.02, 0.02))) +
  labs(title="Observed and Predicted Purchasing Power by Routine Cognitive Value",
       x="Routine Cognitive Value", y="Purchasing Power") +
  theme(panel.grid.major=element_blank())

## The differences are durable

The differences observed across counties were not confined to a specific time period. <a href="#fig-taskarea" class="quarto-xref">Figure 26</a> revealed that the average proportion of employment in each task group remained relatively stable over the fifteen-year panel. The largest discontinuity occurred between 2009 and 2010, coinciding with the Census occupation coding change mentioned in <a href="#sec-data" class="quarto-xref">Section 3</a>. Outside this period, the national distribution of the four task groups remained comparatively stable.

In [83]:
area_df=(df.groupby("year")[["routine_cognitive_share","routine_manual_share",
                             "non_routine_cognitive_share","non_routine_manual_share"]]
           .mean().reset_index()
           .melt(id_vars="year", var_name="group", value_name="share"))
area_names={"routine_cognitive_share":"Routine Cognitive",
            "routine_manual_share":"Routine Manual",
            "non_routine_cognitive_share":"Non-Routine Cognitive",
            "non_routine_manual_share":"Non-Routine Manual"}
area_df["group"]=area_df["group"].map(area_names)
area_df["period"]=np.where(area_df["year"]<=2019, "pre", "post")

# narrow, deliberate gap right at the missing 2020 year, rather than the
# full two year blank span a plain pre/2021 split would otherwise leave
gap_rows=[]
for grp in area_df["group"].unique():
    before=area_df.loc[(area_df["group"]==grp) & (area_df["year"]==2019), "share"].iloc[0]
    after=area_df.loc[(area_df["group"]==grp) & (area_df["year"]==2021), "share"].iloc[0]
    gap_rows.append({"year":2019.9, "group":grp, "share":before+(after-before)*0.45, "period":"pre"})
    gap_rows.append({"year":2020.1, "group":grp, "share":before+(after-before)*0.55, "period":"post"})
area_df=pd.concat([area_df, pd.DataFrame(gap_rows)], ignore_index=True).sort_values("year").reset_index(drop=True)

In [84]:
%%R -i area_df -w 9.5 -h 4.7 -u in -r 150 -b transparent
TASKAREA_COLORS <- TASK_COLORS
group_levels <- c("Non-Routine Cognitive","Non-Routine Manual","Routine Cognitive","Routine Manual")
area_df$group <- factor(area_df$group, levels=group_levels)

pre_df  <- area_df[area_df$period=="pre",]
post_df <- area_df[area_df$period=="post",]

label_year <- 2016
label_x <- 2016.5  # drawn between the 2016 and 2017 ticks rather than on top of either
label_df <- area_df[area_df$year==label_year,]
label_df <- label_df[match(group_levels, label_df$group),]
label_df$cum <- cumsum(label_df$share)
label_df$mid <- label_df$cum - label_df$share/2
# Non-Routine Cognitive and Routine Cognitive are the light members of their hue
# families, so white labels lose contrast there; the two manual groups stay dark.
label_df$text_color <- ifelse(label_df$group %in% c("Non-Routine Manual","Routine Manual"),
                               "white", "#0B0B0B")

ggplot(area_df, aes(x=year, y=share, fill=group)) +
  annotate("rect", xmin=2019.9, xmax=2020.1, ymin=0, ymax=1, fill="grey60", alpha=0.35) +
  geom_area(data=pre_df, stat="identity", position=position_stack(reverse=TRUE), linewidth=0) +
  geom_area(data=post_df, stat="identity", position=position_stack(reverse=TRUE), linewidth=0) +
  geom_text(data=label_df, aes(x=label_x, y=mid, label=group, color=text_color),
            hjust=0.5, size=3.3, fontface="bold") +
  scale_color_identity() +
  geom_vline(xintercept=2009.5, linetype="dashed", linewidth=1.0, color="#0B0B0B") +
  # both labels sit above the panel rather than on top of the top band, which
  # needs clip="off" on the coord below and headroom in the top plot margin
  annotate("text", x=2009.5, y=1.02, label="Census coding change", hjust=0.5, vjust=0, size=3.1, fontface="bold", color="#0B0B0B") +
  annotate("text", x=2020, y=1.02, label="No data", hjust=0.5, vjust=0, size=3.1, fontface="bold", color="#0B0B0B") +
  scale_fill_manual(values=TASKAREA_COLORS, guide="none") +
  # no limits= on either scale: it drops rows outside the range rather than
  # zooming, and the stack tops land a hair above 1.0 in floating point, which
  # punched gaps in the top band. expand=c(0,0) sets the areas flush to both axes.
  scale_y_continuous(labels=function(v) sprintf("%.0f%%", v*100), breaks=seq(0, 1, 0.25),
                      expand=c(0, 0)) +
  scale_x_continuous(breaks=2008:2023, expand=c(0, 0)) +
  coord_cartesian(ylim=c(0, 1), xlim=c(2008, 2023), clip="off") +
  labs(title="Average County Task Group Value by Year",
       x="Year", y="Mean Group Value Across Counties") +
  # y tick labels lifted slightly so the 0% label clears the 2008 label in the
  # bottom-left corner; gridlines are off here, so nothing falls out of alignment
  theme(axis.text.x=element_text(angle=0, hjust=0.5),
        axis.text.y=element_text(vjust=0.15),
        panel.grid.major=element_blank(), panel.grid.minor=element_blank(),
        # right margin widened only enough for the 2023 tick label, which sat half
        # outside the plot. The panel itself is untouched, so the bands stay flush
        # to the axis with no whitespace added inside the plot area
        plot.margin=margin(26, 16, 5.5, 5.5))

County rankings provided a more robust test of whether the same places maintained distinct characteristics over time. <a href="#tbl-rankcorr" class="quarto-xref">Table 9</a> compared each county’s task group ranking in 2010 with its ranking in 2023 using Spearman correlations. Routine manual and non-routine cognitive work exhibited the greatest stability, indicating that counties with relatively high values in 2010 generally maintained high values in 2023. Non-routine manual work showed more movement, while routine cognitive work experienced the largest changes. This ordering aligns with the variance decomposition in <a href="#sec-analysis" class="quarto-xref">Section 4</a>, which demonstrated that routine cognitive work exhibited more within-county variation compared to the other task groups.

In [85]:
from scipy.stats import spearmanr

wide_2010=df[df["year"]==2010].set_index("county_fips")
wide_2023=df[df["year"]==2023].set_index("county_fips")
common=wide_2010.index.intersection(wide_2023.index)

rank_rows=[]
for col, name in area_names.items():
    rho=spearmanr(wide_2010.loc[common, col], wide_2023.loc[common, col]).statistic
    rank_rows.append({"Task Group":name, "Rank Correlation":rho})
rankcorr_df=pd.DataFrame(rank_rows)

style_table(GT(rankcorr_df)
  .fmt_number(columns="Rank Correlation", decimals=2)
  .cols_align(align="center", columns="Rank Correlation")
  .cols_label(**{"Rank Correlation": html("Rank<br>Correlation")})
  .cols_width(cases={"Task Group": "140px", "Rank Correlation": "110px"})
  .tab_source_note("Computed from 2010 onward to avoid the Census occupation coding change."))

/opt/anaconda3/envs/capstone/lib/python3.12/site-packages/great_tables/_render_checks.py:37: RenderWarning: Rendering table with .cols_width() in Quarto may result in unexpected behavior. This is because Quarto performs custom table processing. Either use all percentage widths, or set .tab_options(quarto_disable_processing=True) to disable Quarto table processing.
  warnings.warn(

The stability is visible county by county, not only in the ordering, over the same two years used in <a href="#tbl-rankcorr" class="quarto-xref">Table 9</a> (<a href="#fig-taskchange-map" class="quarto-xref">Figure 27</a>). Routine manual and non-routine manual, the two groups with almost no net movement in <a href="#fig-taskarea" class="quarto-xref">Figure 26</a>, are centered near zero and run in both directions across counties. That mixed direction is consistent with variation sitting between counties rather than within them. Routine cognitive and non-routine cognitive, the pair whose national levels moved most, show a change that is both larger and far more uniform in direction, which matches the within county movement identified in the variance decomposition of <a href="#sec-analysis" class="quarto-xref">Section 4</a>.

In [86]:
import geopandas as gpd

change_wide=(wide_2023.loc[common, list(area_names.keys())]
             -wide_2010.loc[common, list(area_names.keys())])*100
change_wide.columns=[area_names[c] for c in change_wide.columns]
change_long=change_wide.reset_index().melt(id_vars="county_fips", var_name="group", value_name="change")
change_long["fips"]=change_long["county_fips"].astype(str).str.zfill(5)

gdf_change=gpd.read_file("https://www2.census.gov/geo/tiger/GENZ2021/shp/cb_2021_us_county_500k.zip")
gdf_change=gdf_change.rename(columns={"GEOID":"fips"})
gdf_change=gdf_change[~gdf_change["STATEFP"].isin(["02","15","60","66","69","72","78"])]

# cross every continental county with all four groups so counties outside the panel
# still render, grey, in every facet, matching the coverage convention in fig-afford-map
groups_order=list(area_names.values())
county_group_grid=gdf_change[["fips","geometry"]].merge(
    pd.DataFrame({"group":groups_order}), how="cross")
merged_change=county_group_grid.merge(change_long[["fips","group","change"]],
                                      on=["fips","group"], how="left")
os.makedirs("output", exist_ok=True)
merged_change.to_file("output/taskchange_map.geojson", driver="GeoJSON")

change_bound=float(change_long["change"].abs().quantile(0.98))

In [87]:
%%R -i change_long -w 9 -h 5.5 -u in -r 150 -b transparent
change_long$group <- factor(change_long$group,
    levels=rev(c("Routine Cognitive","Routine Manual","Non-Routine Cognitive","Non-Routine Manual")))
change_med <- aggregate(change ~ group, change_long, median)
change_bound_strip <- max(abs(change_long$change), na.rm=TRUE) * 1.05

ggplot(change_long, aes(x=change, y=group, color=group)) +
  geom_vline(xintercept=0, linewidth=0.4, color="#0B0B0B") +
  geom_jitter(height=0.32, width=0, size=1.1, alpha=0.35) +
  geom_point(data=change_med, aes(x=change, y=group), inherit.aes=FALSE,
             shape="|", size=10, color="#0B0B0B", stroke=1.5) +
  scale_color_manual(values=TASK_COLORS, guide="none") +
  scale_x_continuous(limits=c(-change_bound_strip, change_bound_strip),
                      labels=function(v) sprintf("%+.0fpp", v)) +
  labs(title="County Task Group Change, 2010 to 2023",
       x="Percentage Point Change", y=NULL) +
  theme(panel.grid.major.y=element_blank())

<a href="#fig-taskchange-map" class="quarto-xref">Figure 27</a> illustrated the magnitude and direction of these changes county by county. Routine manual and non-routine manual changes remained centered near zero and occurred in both directions, while routine cognitive and non-routine cognitive showed larger and more consistently directed changes. <a href="#fig-taskchange-arrows" class="quarto-xref">Figure 28</a> summarized these movements using the same cognitive to manual and routine to non-routine dimensions introduced in <a href="#fig-taskframework" class="quarto-xref">Figure 1</a>. Most counties shifted towards non-routine cognitive work between 2010 and 2023, while a smaller group moved away from it towards the three task groups associated with lower purchasing power. The arrow figure is particularly valuable as it establishes a direct connection between the observed county changes and the conceptual framework introduced earlier in the study.

In [88]:
# shares the 2010 to 2023 window with tbl-rankcorr and fig-taskchange-map: it covers
# more counties than a 2008 start (a county needs data at both endpoints to have a
# change) and avoids the Census occupation coding change
manual_change_arrows=change_wide["Routine Manual"]+change_wide["Non-Routine Manual"]
nonroutine_change_arrows=change_wide["Non-Routine Cognitive"]+change_wide["Non-Routine Manual"]

arrow_shift_df=pd.DataFrame({
    "fips": change_wide.index.astype(str).str.zfill(5),
    "manual_change": manual_change_arrows.values,
    "nonroutine_change": nonroutine_change_arrows.values,
})
arrow_shift_df["shift_magnitude"]=np.sqrt(arrow_shift_df["manual_change"]**2+arrow_shift_df["nonroutine_change"]**2)

# risk direction relative to the regression's reference group: non-routine cognitive is the
# only one of the four task groups positively associated with purchasing power, so a county
# moving away from it (a negative change in its share) is moving toward the three groups that
# are all associated with lower purchasing power
arrow_shift_df["risk_direction"]=np.where(
    change_wide["Non-Routine Cognitive"].values<0,
    "Away From Non-Routine Cognitive", "Toward Non-Routine Cognitive")

# cap the top 5 percent of shifts so a handful of outlier counties don't dominate the figure;
# direction is unchanged, only the arrow length is scaled back for those counties
mag_cap=float(arrow_shift_df["shift_magnitude"].quantile(0.95))
scale_factor=np.where(arrow_shift_df["shift_magnitude"]>mag_cap,
                      mag_cap/arrow_shift_df["shift_magnitude"], 1.0)
arrow_shift_df["manual_change_capped"]=arrow_shift_df["manual_change"]*scale_factor
arrow_shift_df["nonroutine_change_capped"]=arrow_shift_df["nonroutine_change"]*scale_factor

gdf_arrows=gdf_change.to_crs(5070)
gdf_arrows["cx"]=gdf_arrows.geometry.centroid.x
gdf_arrows["cy"]=gdf_arrows.geometry.centroid.y

arrow_data=gdf_arrows[["fips","cx","cy"]].merge(arrow_shift_df, on="fips", how="inner")

In [89]:
%%R -i arrow_data -w 9.5 -h 6.5 -u in -r 150 -b transparent
suppressMessages(library(sf))

ARROW_SCALE <- 8000
arrow_data$xend <- arrow_data$cx + arrow_data$manual_change_capped * ARROW_SCALE
arrow_data$yend <- arrow_data$cy + arrow_data$nonroutine_change_capped * ARROW_SCALE
arrow_data$risk_direction <- factor(arrow_data$risk_direction,
    levels=c("Away From Non-Routine Cognitive","Toward Non-Routine Cognitive"))

counties_arrows <- st_read("output/taskchange_map.geojson", quiet=TRUE)
counties_arrows <- counties_arrows[!duplicated(counties_arrows$fips),]

ggplot() +
  geom_sf(data=counties_arrows, fill="#FCFCFB", color="#E1E0D9", linewidth=0.08) +
  geom_segment(data=arrow_data,
               aes(x=cx, y=cy, xend=xend, yend=yend, color=risk_direction, linewidth=shift_magnitude),
               arrow=arrow(length=unit(1.3,"mm"), type="closed"), lineend="round", alpha=0.75) +
  scale_linewidth_continuous(range=c(0.2, 1.0), guide="none") +
  # Solid red and green, per Julian 14 Aug 2026, replacing an earlier orange and
  # purple pair. Green marks movement toward non-routine cognitive work, the
  # reference group associated with higher purchasing power.
  scale_color_manual(values=c("Away From Non-Routine Cognitive"=RED,
                              "Toward Non-Routine Cognitive"=GREEN), name=NULL) +
  coord_sf(crs=st_crs(5070), datum=NA) +
  theme_void(base_size=11) +
  labs(title="Direction of County Task Group Shift, 2010 to 2023") +
  theme(legend.position="bottom",
        plot.title=element_text(face="bold", size=13, hjust=0.5),
        plot.background=element_rect(fill=NA, color=NA))

Finally, the estimated associations themselves remained consistent over time. <a href="#fig-stability" class="quarto-xref">Figure 29</a> compared the full panel with separate specifications for 2010 to 2019 and 2021 to 2023. The ordering of the three task group coefficients remained unchanged across all three periods, and their magnitudes shifted only slightly. Reporting the coefficients as percentage changes ensured comparability between the periods, even though nominal purchasing power increased later in the panel. The persistence of both the county patterns and the coefficient ordering suggested that the association was not confined to a single recession, recovery, or pandemic period.

In [90]:
def fit_window_log(frame):
    Xw=pd.concat([frame[res_cols],
                  pd.get_dummies(frame["year"], prefix="year", drop_first=True).astype(float)], axis=1)
    Xw=sm.add_constant(Xw)
    return sm.OLS(np.log(frame["affordability_salary"]), Xw).fit(
        cov_type="cluster", cov_kwds={"groups": frame["county_fips"]})

windows={"Full panel": reg,
         "2010 to 2019": reg[(reg["year"]>=2010)&(reg["year"]<=2019)],
         "2021 to 2023": reg[reg["year"]>=2021]}

stab_rows=[]
for term, name in zip(group_terms, group_names):
    row={"Task group":name}
    for wname, frame in windows.items():
        b=fit_window_log(frame).params[term]
        row[wname]=(np.exp(b*0.01)-1)*100
    stab_rows.append(row)
stability_df=pd.DataFrame(stab_rows)

stab_long=stability_df.melt(id_vars="Task group", var_name="window", value_name="value")
range_df=stability_df.assign(
    lo=stability_df[["Full panel","2010 to 2019","2021 to 2023"]].min(axis=1),
    hi=stability_df[["Full panel","2010 to 2019","2021 to 2023"]].max(axis=1),
)[["Task group","lo","hi"]]

In [91]:
%%R -i stab_long -i range_df -w 9 -h 4 -u in -r 150 -b transparent
BLACK <- "#252525"

group_order <- c("Routine Manual", "Routine Cognitive", "Non-Routine Manual")
stab_long$window <- factor(stab_long$window, levels=c("2010 to 2019", "Full panel", "2021 to 2023"))
stab_long$`Task group` <- factor(stab_long$`Task group`, levels=group_order)
range_df$`Task group` <- factor(range_df$`Task group`, levels=group_order)

window_colors <- c("2010 to 2019"=BLUE, "Full panel"=BLACK, "2021 to 2023"=ORANGE)
window_shapes <- c("2010 to 2019"=16, "Full panel"=18, "2021 to 2023"=16)
window_sizes  <- c("2010 to 2019"=3.2, "Full panel"=4.6, "2021 to 2023"=3.2)

# the three windows sit within 0.04 of each other for some groups, so stacking all
# three value labels at one height would overlap. Place them by rank instead:
# leftmost reads left, rightmost reads right, and the middle one sits above.
stab_long$rank <- ave(stab_long$value, stab_long$`Task group`, FUN=rank)
lab_pad <- diff(range(stab_long$value)) * 0.03

ggplot() +
  geom_vline(xintercept=0, linetype="dashed", linewidth=0.5, color=BLACK) +
  geom_segment(data=range_df, aes(x=lo, xend=hi, y=`Task group`, yend=`Task group`,
                                  linewidth="Range across windows"),
               color="#D6D6D2", lineend="round") +
  geom_point(data=stab_long, aes(x=value, y=`Task group`, color=window, shape=window, size=window)) +
  geom_text(data=subset(stab_long, rank==1),
            aes(x=value - lab_pad, y=`Task group`, label=sprintf("%.2f", value), color=window),
            hjust=1, size=3.1, fontface="bold", show.legend=FALSE) +
  geom_text(data=subset(stab_long, rank==3),
            aes(x=value + lab_pad, y=`Task group`, label=sprintf("%.2f", value), color=window),
            hjust=0, size=3.1, fontface="bold", show.legend=FALSE) +
  geom_text(data=subset(stab_long, rank==2),
            aes(x=value, y=`Task group`, label=sprintf("%.2f", value), color=window),
            vjust=-1.5, size=3.1, fontface="bold", show.legend=FALSE) +
  scale_color_manual(values=window_colors, name=NULL,
                     labels=c("2010 to 2019","Full panel (2008 to 2023)","2021 to 2023")) +
  scale_shape_manual(values=window_shapes, name=NULL,
                     labels=c("2010 to 2019","Full panel (2008 to 2023)","2021 to 2023")) +
  scale_size_manual(values=window_sizes, guide="none") +
  scale_linewidth_manual(values=c("Range across windows"=3.2), name=NULL) +
  scale_x_continuous(expand=expansion(mult=c(0.10, 0.06))) +
  labs(title="Coefficient Stability Across Time Windows",
       x="Percent Change per One Percentage Point Shift From Non-Routine Cognitive Work",
       y=NULL) +
  theme(legend.position="bottom",
        axis.text.y=element_text(face="bold", size=9.5),
        panel.grid.major.y=element_blank())

The results revealed that county task groups continued to be linked to purchasing power even after accounting for factors like poverty, unemployment, population, and the year. The largest difference was observed between non-routine manual work and other tasks. A one percentage point shift from non-routine cognitive work to non-routine manual work was associated with a reduction in purchasing power of approximately \$846 at the panel mean, with a 95 percent confidence interval of ±\$38. Routine cognitive and routine manual work were also negatively associated with purchasing power compared to non-routine cognitive work. Among the variables examined, poverty showed the strongest association with purchasing power, with each additional percentage point associated with a reduction of approximately 2.9 percent. The random forest analysis indicated that task groups provided additional information beyond the economic controls. Removing the task groups and refitting the model resulted in a decrease in the held out R² value of 0.042, while removing poverty and unemployment reduced it by 0.178. Jointly permuting the task groups further reduced the R² value by 0.153, compared to 0.128 for year indicators and 0.008 for state indicators. The geographic analysis showed that counties dominated by non-routine cognitive work had higher purchasing power, while counties dominated by the two manual groups had lower purchasing power. The stability analysis demonstrated that these county differences persisted over time. Routine manual and non-routine cognitive work maintained the most consistent county ordering between 2010 and 2023, while routine cognitive work exhibited the greatest movement. Interestingly, the ordering of the task group coefficients remained unchanged in the full panel, the 2010 to 2019 period, and the 2021 to 2023 period.

The level model, log linear model, random forest, and neural network were employed to determine the most suitable representation of the relationship. The level specification established negative task group associations but revealed systematic curvature in the residuals, suggesting that a constant dollar difference across the purchasing power distribution was inadequate. Logging purchasing power mitigated residual skew from 1.01 to 0.09, resulting in an in-sample R² of 0.913 and a held-out R² of 0.896. The random forest captured the nonlinear structure evident in the level model but failed to enhance held-out performance. Its log target specification achieved a held-out R² of 0.877, 0.019 below the log linear model, while the raw target forest achieved a training R² of 0.989 but only 0.876 on held-out counties. The neural network introduced greater flexibility, but regularization did not yield an improvement over either retained model. Five-fold grouped cross-validation yielded similar general comparisons, with the difference between the log linear and random forest models remaining smaller than the variation across folds. Across these four specifications, the log linear model maintained task group associations, rectified most of the diagnostic issues in the level model, and achieved the highest held-out performance. The implications and limitations of these results are discussed in the section titled “<a href="#sec-conclusions" class="quarto-xref">Section 6</a>.”

# Conclusions

## Summary of findings

Across all counties analyzed over a span of fifteen years, county task groups remained linked to purchasing power after accounting for factors like poverty, unemployment, population, and the specific year. The largest difference in task group performance was observed in non-routine manual work. A one percentage point shift from non-routine cognitive to non-routine manual work was associated with approximately \$846 lower purchasing power on average, with a 95 percent confidence interval of ±\$38. Routine cognitive and routine manual work were also negatively correlated with purchasing power compared to non-routine cognitive work. Among the variables examined, poverty demonstrated the strongest association with purchasing power, with each additional percentage point associated with approximately 2.9 percent lower purchasing power. Despite these conventional economic measures, the task groups still provided valuable insights. Removing them and refitting the random forest reduced the held out R² by 0.042, while jointly permuting them reduced it by 0.153. In comparison, permuting the year indicators reduced R² by 0.128, and permuting the state indicators reduced it by only 0.008.

The sequence of models provided insights into the most suitable representation of the relationship. The level specification identified negative associations but revealed systematic curvature in the residuals, suggesting that a constant dollar relationship was inadequate for the data. Logging purchasing power reduced the residual skew from 1.01 to 0.09, resulting in an in-sample R² of 0.913 and a held-out R² of 0.896. The random forest analysis explored whether nonlinear relationships and interactions improved upon this specification, but its log target version achieved a held-out R² of 0.877. The raw target forest also demonstrated a closer fit to the training data compared to unseen counties, with R² values of 0.989 and 0.876, respectively. The neural network introduced even greater flexibility but failed to yield further improvements after regularization. Five-fold grouped cross-validation yielded similar general comparisons, with differences between the log linear model and random forest being smaller than the variation across folds. Geographic and stability analyses extended these results beyond model performance by demonstrating that county task group differences persisted from 2010 to 2023 and that the ordering of the task group coefficients remained consistent across the entire panel, the 2010 to 2019 period, and the 2021 to 2023 period.

The study also clarified the scope of its findings regarding automation and affordability. The analysis did not measure automation adoption, job displacement, or the likelihood of specific counties losing jobs to new technologies. Instead, it examined the types of tasks already present in county employment and assessed their association with purchasing power. Counties with higher concentrations of routine and manual work, which are often emphasized in the automation literature, tended to have lower purchasing power compared to counties with more non-routine cognitive work. This relationship persisted after adjustment and remained consistent across geography and time. However, it did not establish a causal link between automation and purchasing power differences. Therefore, the study indirectly addressed automation by identifying regions where task structures commonly associated with higher automation potential coincided with lower purchasing power, leaving the consequences of actual technological displacement for future research.

## Contributions

This study made three main contributions to the existing task literature. First, it shifted the outcome from nominal income to purchasing power. Median household income was adjusted using BEA Regional Price Parities, directly incorporating local price differences into the measure of household resources. This adjustment distinguished counties with similar nominal incomes where the same income supported varying levels of purchasing power. Instead of comparing income alone, the study examined what that income could purchase after accounting for local price variations.

Second, the study measured task groups at the county level and tracked them over time. The final panel comprised 840 counties from 2008 to 2023, excluding 2020, and represented approximately 84 percent of the U.S. population. Using counties provided a finer geographic scale than broader labor market areas and directly connected task groups with local purchasing power, poverty, and unemployment. The panel structure demonstrated that these differences were not confined to a single point in time. County task group patterns remained relatively consistent throughout the study period, enabling the analysis to distinguish enduring differences between counties from short-term changes within them.

The county-level analysis also revealed that task groups contained information beyond conventional measures of local economic conditions. Removing the task groups and refitting the random forest reduced the held-out R² by 0.042, while jointly permuting them reduced it by 0.153. Poverty exhibited a stronger association with purchasing power than any individual task group, but poverty and unemployment did not fully capture the information contained within the task groups. Consequently, counties that appeared similar on these conventional measures could still differ in the types of work residents performed and the purchasing power associated with those differences.

Third, the study aimed to determine the most appropriate representation of the relationship between task groups and purchasing power, rather than assuming a specific functional form beforehand. The level specification revealed systematic residual curvature, while logging purchasing power reduced residual skew from 1.01 to 0.09 and achieved a held-out R² of 0.896. The random forest model achieved 0.877 on the same held-out counties, and a three-hidden-layer neural network provided no further improvement. Therefore, greater model flexibility did not enhance held-out performance once purchasing power was represented proportionally. The contribution was not solely the use of multiple models but their application to test whether a more complex representation of the relationship was supported by the data.

These contributions resulted in a practical framework for comparing local economic conditions. Purchasing power indicated the extent to which household income extended after accounting for local prices, while task groups described the types of work comprising county employment. Consequently, combining the two revealed differences that income, poverty, or unemployment alone could not fully capture. Counties with similar values on conventional economic indicators could still differ in both their task groups and purchasing power. This resulting framework provided policymakers and economic development organizations with a reproducible method to examine local economic conditions using publicly available data.

## Limitations

Five limitations defined the scope of the results. First, the findings were correlational rather than causal. Counties were not randomly assigned to task groups, and the analysis could not isolate task groups from all other characteristics that varied across places. Unobserved factors related to both county employment and purchasing power could therefore contribute to the estimated relationship. The results showed that task groups remained associated with purchasing power after accounting for the measured controls, but they did not establish that differences in task groups caused differences in purchasing power.

Second, the analytical panel was limited to counties above the population threshold required for the occupational data. The final sample included 840 counties, representing approximately 84 percent of the U.S. population, but smaller and more rural counties were underrepresented. Therefore, the estimates described the more populous counties included in the panel rather than all U.S. counties. Relationships in counties below the threshold could differ from those observed in the analytical sample.

Third, the geographic precision of Regional Price Parities varied across counties. A majority of panel observations relied on a state-level price parity rather than a more local measure, making purchasing power less geographically precise for those observations. Consequently, some within-state differences in local prices were not captured by the outcome. The analysis applied these measures consistently but did not separately test whether the association differed according to the geographic precision of the price measure.

Fourth, purchasing power was measured in nominal dollars, which were not adjusted for changes in the national price level over time. Regional Price Parities, on the other hand, corrected for price differences within a given year but did not establish a common scale for dollars from different years. The year indicators absorbed any shared national price change across all counties, leaving the task group and control coefficients unaffected. The year coefficients themselves combined inflation with other national conditions that fluctuated between years, making them unsuitable as a standalone measure of inflation. Consequently, dollar comparisons across years represented nominal amounts rather than constant purchasing power.

Finally, the primary panel regression did not include state indicators, leaving open the possibility that unmeasured state-level differences contributed to the estimated associations. The predictive models provided a partial check by including state indicators alongside task groups, poverty, unemployment, and year. Permuting the state indicators reduced the held-out R² by only 0.008, indicating that they contributed relatively little predictive information in that specification. However, the predictive models and panel regression were not identical, so this result did not eliminate the possibility of state-level confounding in the primary estimates.

## Ethical considerations

The study required clear boundaries on how the results were interpreted. Since the county was the unit of analysis, the findings described places rather than individual workers. Applying county-level relationships to individuals would constitute an ecological fallacy. Consequently, the results could not be used to infer a worker’s purchasing power, economic vulnerability, or likelihood of being affected by automation based on their task group.

Representation also necessitated caution. Counties were excluded because occupational data were unavailable below the source publication threshold, not due to an analysis-driven decision. As a result, smaller and more rural counties were underrepresented in the panel. Extending the findings to those counties would require extrapolation beyond the observed data, which was particularly important when comparing geographic patterns or identifying places that might warrant further attention.

The data also carried a selection bias that the analysis couldn’t eliminate. The publication threshold excluded smaller counties before constructing the analytical sample, resulting in a panel that wasn’t a random representation of all U.S. counties. Consequently, rural and less populous counties were underrepresented, making the results more accurate for the included counties than for those excluded by the threshold. This limitation was particularly significant because smaller rural counties are often central to discussions about automation and local economic vulnerability.

The associational nature of the findings also limited their use for policy decisions. Treating the estimated relationships as causal could lead to the allocation of resources to change a county’s task groups even if an unmeasured economic or geographic factor contributed to the observed difference. While the results supported identifying where lower purchasing power and specific task groups coexisted, they did not establish which intervention would alter those conditions. Similarly, the language used to describe counties required similar care. Labeling places based on presumed automation risk or economic vulnerability could stigmatize communities or deter investment based on relationships that the study did not establish causally.

Finally, the study relied entirely on publicly available aggregate data. No individual-level records or personally identifiable information entered the analytical dataset, and all reported results described county-level patterns rather than individual people.

## Future directions

Three directions emerge from these limitations. The first is the coverage gap, which could potentially be bridged using satellite imagery. Jean et al. ([2016](#ref-Jean2016))’s estimate of local economic conditions from daytime and nighttime imagery, where survey data are scarce, could be applied here. This approach could generate purchasing power estimates for counties below the American Community Survey publication threshold and assess whether the geographic patterns observed in this study extend to areas outside the analytical panel. While it wouldn’t expand the task group association itself, as comparable occupational estimates would still be unavailable for those counties, reproducing the purchasing power geography in smaller and more rural counties would help determine if the spatial patterns reported here reflect the country more broadly rather than the more populous counties included in the sample.

A second direction is to extend the study beyond the 2008-2023 period as additional occupational, income, price, poverty, and unemployment data become available. The current analysis revealed that the ordering of the task group associations remained stable across the entire panel and across the 2010-2019 and 2021-2023 periods. Extending the panel would directly test whether these relationships persist as local labor markets continue to evolve. Additionally, it would allow the results reported here to be evaluated out of time using observations that were not available when the original models were developed. Replicating the direction, magnitude, and proportional form of the associations in later years would provide stronger evidence that the patterns identified in this study were enduring rather than specific to the observed period.

The third approach is to investigate the underlying reasons for the correlation between task groups and purchasing power. Establishing a causal relationship would necessitate identifying a source of variation in task groups that was not influenced by local purchasing power. This could be achieved through factors such as plant openings and closures, major changes in production technology, or identifiable technology adoption shocks. Such an analysis would require a separate study with distinct data and a research design capable of isolating plausible exogenous changes in local task groups. By conducting this analysis, we could go beyond merely confirming the association observed in this study and begin to explore whether changes in the types of work performed within a county are accompanied by shifts in purchasing power.

## Conclusion

This study discovered a consistent link between county task groups and purchasing power. Counties with higher concentrations of routine and manual work generally had lower purchasing power compared to those with more non-routine cognitive work, even after accounting for factors like poverty, unemployment, population, and year. The largest difference was observed for non-routine manual work. At the panel mean, a one percentage point shift from non-routine cognitive to non-routine manual work was associated with approximately \$846 lower purchasing power, with a 95 percent confidence interval of ±\$38.

The relationship persisted across both time and model specifications. County task group patterns remained relatively stable over the fifteen-year panel, and the ordering of the estimated associations remained consistent across the examined periods. Diagnostic testing revealed that the relationship was better represented proportionally than as a constant dollar difference. The log linear specification also achieved stronger held-out performance compared to the random forest, while the neural network provided no further improvement. These results suggested that additional model complexity was unnecessary to capture the primary relationship observed in the data.

The findings did not establish a causal relationship between task groups and purchasing power, nor did the study directly measure automation adoption or job displacement. Instead, the results indicated that the types of work concentrated within a county provided information about purchasing power beyond poverty, unemployment, population, and year. While poverty remained an important indicator of local economic conditions, it did not fully explain the association between task groups and purchasing power. Counties with higher concentrations of routine and manual work consistently exhibited lower purchasing power across geography and time. Therefore, understanding differences in purchasing power across counties required considering not only household income and local prices, but also the types of work that characterized the local economy.

# References

Abadi, Martín, Ashish Agarwal, Paul Barham, Eugene Brevdo, Zhifeng Chen, Craig Citro, Greg S. Corrado, et al. 2016. “TensorFlow: Large-Scale Machine Learning on Heterogeneous Distributed Systems.” <https://www.tensorflow.org/>.

Acemoglu, Daron, and David Autor. 2011. “Skills, Tasks and Technologies: Implications for Employment and Earnings.” In *Handbook of Labor Economics*, edited by David Card and Orley Ashenfelter, 4:1043–1171. Elsevier. <https://doi.org/10.1016/S0169-7218(11)02410-5>.

Autor, David H., and David Dorn. 2013. “The Growth of Low-Skill Service Jobs and the Polarization of the US Labor Market.” *American Economic Review* 103 (5): 1553–97. <https://doi.org/10.1257/aer.103.5.1553>.

Autor, David H., Frank Levy, and Richard J. Murnane. 2003. “The Skill Content of Recent Technological Change: An Empirical Exploration.” *The Quarterly Journal of Economics* 118 (4): 1279–1333. <https://doi.org/10.1162/003355303322552801>.

Chiripanhura, Blessing. 2011. “Median and Mean Income Analyses: Their Implications for Material Living Standards and National Well-Being.” *Economic & Labour Market Review* 5: 45–63. <https://doi.org/10.1057/elmr.2011.17>.

Harris, Charles R., K. Jarrod Millman, Stéfan J. van der Walt, Ralf Gommers, Pauli Virtanen, David Cournapeau, Eric Wieser, et al. 2020. “Array Programming with NumPy.” *Nature* 585 (7825): 357–62. <https://doi.org/10.1038/s41586-020-2649-2>.

Jean, Neal, Marshall Burke, Michael Xie, W. Matthew Davis, David B. Lobell, and Stefano Ermon. 2016. “Combining Satellite Imagery and Machine Learning to Predict Poverty.” *Science* 353 (6301): 790–94. <https://doi.org/10.1126/science.aaf7894>.

Jordahl, Kelsey, Joris Van den Bossche, Martin Fleischmann, Jacob Wasserman, James McBride, Jeffrey Gerard, Jeff Tratner, et al. 2020. “Geopandas/Geopandas.” <https://doi.org/10.5281/zenodo.3946761>.

McKinney, Wes. 2010. “Data Structures for Statistical Computing in Python.” In *Proceedings of the 9th Python in Science Conference*, 56–61. <https://doi.org/10.25080/Majora-92bf1922-00a>.

Moretti, Enrico. 2013. “Real Wage Inequality.” *American Economic Journal: Applied Economics* 5 (1): 65–103. <https://doi.org/10.1257/app.5.1.65>.

Pebesma, Edzer. 2018. “Simple Features for R: Standardized Support for Spatial Vector Data.” *The R Journal* 10 (1): 439–46. <https://doi.org/10.32614/RJ-2018-009>.

Pedersen, Thomas Lin. 2024. *Patchwork: The Composer of Plots*. <https://CRAN.R-project.org/package=patchwork>.

Pedregosa, Fabian, Gaël Varoquaux, Alexandre Gramfort, Vincent Michel, Bertrand Thirion, Olivier Grisel, Mathieu Blondel, et al. 2011. “Scikit-Learn: Machine Learning in Python.” *Journal of Machine Learning Research* 12: 2825–30. <https://jmlr.org/papers/v12/pedregosa11a.html>.

R Core Team. 2024. *R: A Language and Environment for Statistical Computing*. Vienna, Austria: R Foundation for Statistical Computing. <https://www.R-project.org/>.

Saad, Lydia. 2023. “More U.S. Workers Fear Technology Making Their Jobs Obsolete.” Gallup. <https://news.gallup.com/poll/510551/workers-fear-technology-making-jobs-obsolete.aspx>.

Seabold, Skipper, and Josef Perktold. 2010. “Statsmodels: Econometric and Statistical Modeling with Python.” In *Proceedings of the 9th Python in Science Conference*, 92–96. <https://doi.org/10.25080/Majora-92bf1922-011>.

Smith, Aaron, and Monica Anderson. 2017. “Automation in Everyday Life.” Pew Research Center. <https://www.pewresearch.org/internet/2017/10/04/automation-in-everyday-life/>.

U.S. Bureau of Economic Analysis. 2024. “Regional Price Parities by State and Metro Area.” <https://www.bea.gov/data/prices-inflation/regional-price-parities-state-and-metro-area>.

U.S. Bureau of Labor Statistics. 2024. “Local Area Unemployment Statistics.” <https://www.bls.gov/lau/>.

U.S. Census Bureau. 2023a. “Core Based Statistical Area Delineation Files.” <https://www.census.gov/geographies/reference-files/time-series/demo/metro-micro/delineation-files.html>.

———. 2023b. “Vintage 2023 Population Estimates.” <https://www.census.gov/programs-surveys/popest.html>.

———. 2024a. “American Community Survey 1-Year Estimates.” <https://www.census.gov/programs-surveys/acs>.

———. 2024b. “Small Area Income and Poverty Estimates.” <https://www.census.gov/programs-surveys/saipe.html>.

Virtanen, Pauli, Ralf Gommers, Travis E. Oliphant, Matt Haberland, Tyler Reddy, David Cournapeau, Evgeni Burovski, et al. 2020. “SciPy 1.0: Fundamental Algorithms for Scientific Computing in Python.” In *Nature Methods*, 17:261–72. <https://doi.org/10.1038/s41592-019-0686-2>.

Wickham, Hadley. 2016. *Ggplot2: Elegant Graphics for Data Analysis*. Springer-Verlag New York. <https://ggplot2-book.org/>.

Wickham, Hadley, Thomas Lin Pedersen, and Dana Seidel. 2023. *Scales: Scale Functions for Visualization*. <https://CRAN.R-project.org/package=scales>.

## Software

Data processing, modeling, and machine learning were carried out in Python using pandas ([McKinney 2010](#ref-pandas2010)), NumPy ([Harris et al. 2020](#ref-numpy2020)), SciPy ([Virtanen et al. 2020](#ref-scipy2020)), statsmodels ([Seabold and Perktold 2010](#ref-statsmodels2010)), scikit-learn ([Pedregosa et al. 2011](#ref-sklearn2011)), TensorFlow ([Abadi et al. 2016](#ref-tensorflow2016)), and GeoPandas ([Jordahl et al. 2020](#ref-geopandas2020)). Figures were produced in R ([R Core Team 2024](#ref-rcoreteam2024)) using ggplot2 ([Wickham 2016](#ref-ggplot2_2016)), patchwork ([Pedersen 2024](#ref-patchwork2024)), scales ([Wickham, Pedersen, and Seidel 2023](#ref-scales2023)), and sf ([Pebesma 2018](#ref-sf2018)).